# Instance-Wise Unlearning trên máy ảo

Notebook triển khai `PLAN.md` cho dataset CSV local và checkpoint trong repo:

- AOL CSV: known (0), thư mục con `icloud`;
- 100 Top Search: unknown (1), thư mục con `icloud_100`;
- `weight_encoder_trained`: checkpoint `pretrain_AOL.pth`.

Data contract:

```text
one CSV row
-> 10,000 x [label_id, relative_time, direction, packet_size]
-> features [10,000, 3] + packet mask
-> pretrained AOL encoder + binary MLP
-> known (0) / unknown (1)
```

Một CSV row là một instance. Instance được định danh bằng source, split file,
line number và SHA-256 của row.

Thứ tự chạy:

1. Kiểm tra đường dẫn dataset local, checkpoint và CUDA.
2. Run All với `RUN_PIPELINE = False` để kiểm tra path/config.
3. Kiểm tra `NOTEBOOK_COMMANDS` ở cell cuối, rồi đặt `RUN_PIPELINE = True`.
4. Cell cuối chạy tuần tự `validate-schema`, `train-base`, `make-manifests`, `unlearn`.

Dataset mặc định nằm dưới `/home/ubuntu/Documents/KF/data_processed`. Artifact được ghi dưới
`<repo>/artifacts/vm-training/instance-wise`; có thể override bằng biến môi trường.


## 1. Imports, constants, records, and common helpers


In [ ]:
"""Instance-wise unlearning for processed traffic CSV data on a virtual machine.

The notebook supports four explicit phases:

* ``validate-schema``: resolve and validate both local CSV datasets.
* ``train-base``: train and save the binary encoder+MLP checkpoint.
* ``make-manifests``: select nested AOL training instances to forget.
* ``unlearn``: benchmark independent scope baselines and adapted paper methods.

Raw PCAP files are not read. A model instance is one physical CSV row.
"""

from __future__ import annotations

import argparse
import copy
import csv
import hashlib
import json
import math
import os
import random
import shutil
import time
from collections import Counter, defaultdict
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Any, Iterable, Iterator, Sequence

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch import Tensor
from torch.utils.data import DataLoader, Dataset


def resolve_repo_root() -> Path:
    override = os.environ.get("BKCS_REPO_ROOT")
    start = Path(override).expanduser().resolve() if override else Path.cwd().resolve()
    marker = Path("experiments/instance-wise-unlearning/virtual-machine/pipeline.ipynb")
    for candidate in (start, *start.parents):
        if (candidate / marker).is_file():
            return candidate
    raise FileNotFoundError(
        "Cannot find repo root; run inside the repository or set BKCS_REPO_ROOT"
    )


REPO_ROOT = resolve_repo_root()
VM_INPUT_ROOT = Path(
    os.environ.get("BKCS_VM_INPUT_ROOT", "/home/ubuntu/Documents/KF/data_processed")
).expanduser().resolve()
VM_WORKING_ROOT = Path(
    os.environ.get(
        "BKCS_VM_WORKING_ROOT",
        str(REPO_ROOT / "artifacts/vm-training"),
    )
).expanduser().resolve()
OUTPUT_ROOT = VM_WORKING_ROOT / "instance-wise"
REQUIRED_DATASET_FILES = {"train_data.csv", "test_data.csv", "label_mapping.csv"}


def resolve_dataset_dir(folder_name: str) -> Path:
    """Resolve one local dataset folder by schema."""
    if not VM_INPUT_ROOT.is_dir():
        raise FileNotFoundError(f"VM input root does not exist: {VM_INPUT_ROOT}")
    matches = [
        path.resolve()
        for path in VM_INPUT_ROOT.rglob(folder_name)
        if path.is_dir()
        and REQUIRED_DATASET_FILES.issubset({item.name for item in path.iterdir()})
    ]
    unique = sorted(set(matches))
    if len(unique) != 1:
        raise FileNotFoundError(
            f"Expected exactly one {folder_name!r} directory containing "
            f"{sorted(REQUIRED_DATASET_FILES)}; found: {unique}"
        )
    return unique[0]


def ensure_under(path: Path, root: Path, label: str) -> Path:
    resolved = path.expanduser().resolve()
    resolved.relative_to(root.resolve())
    if label == "output" and resolved == root.resolve():
        raise ValueError(f"Output path must be below {root}")
    return resolved


DEFAULT_AOL_DIR = (
    os.environ["AOL_DATASET_DIR"]
    if "AOL_DATASET_DIR" in os.environ
    else str(resolve_dataset_dir("icloud"))
)
DEFAULT_UNKNOWN_DIR = (
    os.environ["UNKNOWN_100_DATASET_DIR"]
    if "UNKNOWN_100_DATASET_DIR" in os.environ
    else str(resolve_dataset_dir("icloud_100"))
)
DEFAULT_PRETRAIN = os.environ.get(
    "PRETRAIN_AOL_PATH",
    str(REPO_ROOT / "experiments/instance-wise-unlearning/virtual-machine/weights/pretrain_AOL.pth"),
)

MAX_PACKETS = 10_000
RAW_FIELDS_PER_PACKET = 4
PACKET_FEATURES = 3
ROW_WIDTH = MAX_PACKETS * RAW_FIELDS_PER_PACKET
EMBEDDING_DIM = 256
NUM_CLASSES = 2
KNOWN = 0
UNKNOWN = 1
INDEX_VERSION = "instance-wise-vm-csv-index-v1"
FEATURE_TRANSFORM = "legacy_raw_v1"
DEFAULT_BENCHMARK_MODES = (
    "baseline",
    "only_head",
    "full_mlp",
    "last_encoder_block",
    "full_model",
    "amnesiac",
    "blindspot",
    "bnd",
    "mkmmd",
)
LEGACY_METHODS = (
    "relabel_only",
    "retain_rehearsal",
    "l2ul_mas",
    "l2ul_adv",
    "l2ul_adv_mas",
)
MODE_SPECS = {
    "baseline": {
        "objective": "baseline",
        "trainable_scope": "none",
        "adaptation": "base checkpoint without parameter updates",
    },
    "only_head": {
        "objective": "targeted_rehearsal",
        "trainable_scope": "only_head",
        "adaptation": "update only the final classifier layer",
    },
    "full_mlp": {
        "objective": "targeted_rehearsal",
        "trainable_scope": "full_mlp",
        "adaptation": "update the complete MLP classifier",
    },
    "last_encoder_block": {
        "objective": "targeted_rehearsal",
        "trainable_scope": "last_encoder_block",
        "adaptation": "update the final encoder block and MLP",
    },
    "full_model": {
        "objective": "targeted_rehearsal",
        "trainable_scope": "full_model",
        "adaptation": "update the full encoder and MLP",
    },
    "amnesiac": {
        "objective": "amnesiac",
        "trainable_scope": "full_mlp",
        "paper": "https://doi.org/10.1609/aaai.v35i13.17371",
        "adaptation": "gradient ascent on Df plus retain rehearsal; no stored batch updates",
    },
    "blindspot": {
        "objective": "blindspot",
        "trainable_scope": "full_model",
        "paper": "https://doi.org/10.1609/aaai.v37i6.25879",
        "adaptation": "base teacher on Dr and soft unknown teacher on Df",
    },
    "bnd": {
        "objective": "bnd",
        "trainable_scope": "full_mlp",
        "paper": "https://openaccess.thecvf.com/content/CVPR2023/html/Chen_Boundary_Unlearning_Rapid_Forgetting_of_Deep_Networks_via_Shifting_the_CVPR_2023_paper.html",
        "adaptation": "instance-wise known-to-unknown logit margin",
    },
    "mkmmd": {
        "objective": "mkmmd",
        "trainable_scope": "full_model",
        "paper": "https://www.jmlr.org/papers/v13/gretton12a.html",
        "adaptation": "align Df embeddings to unknown and preserve Dr embeddings",
    },
}
DEFAULT_MODE_HYPERPARAMETERS = {
    # Scope baselines use increasingly conservative learning rates as the scope widens.
    "baseline": {
        "epochs": 0, "learning_rate": 0.0, "weight_decay": 0.0,
        "max_steps_per_epoch": 0, "lambda_forget": 0.0, "lambda_retain": 0.0,
        "min_epochs": 0, "early_stop_forget_success": 1.0,
    },
    "only_head": {
        "epochs": 8, "learning_rate": 3e-4, "weight_decay": 1e-4,
        "max_steps_per_epoch": 10, "lambda_forget": 2.0, "lambda_retain": 1.0,
        "min_epochs": 3, "early_stop_forget_success": 1.0,
    },
    "full_mlp": {
        "epochs": 4, "learning_rate": 7.5e-5, "weight_decay": 1e-4,
        "max_steps_per_epoch": 10, "lambda_forget": 1.0, "lambda_retain": 2.0,
        "min_epochs": 2, "early_stop_forget_success": 1.0,
    },
    "last_encoder_block": {
        "epochs": 3, "learning_rate": 5e-5, "weight_decay": 1e-4,
        "max_steps_per_epoch": 10, "lambda_forget": 1.0, "lambda_retain": 2.0,
        "min_epochs": 2, "early_stop_forget_success": 1.0,
    },
    "full_model": {
        "epochs": 3, "learning_rate": 2.5e-5, "weight_decay": 1e-4,
        "max_steps_per_epoch": 10, "lambda_forget": 1.0, "lambda_retain": 2.5,
        "min_epochs": 2, "early_stop_forget_success": 1.0,
    },
    # The adapted negative-gradient objective needs a longer budget and stronger forget term.
    "amnesiac": {
        "epochs": 12, "learning_rate": 3e-4, "weight_decay": 1e-4,
        "max_steps_per_epoch": 10, "lambda_forget": 5.0, "lambda_retain": 0.5,
        "min_epochs": 3, "early_stop_forget_success": 1.0,
    },
    "blindspot": {
        "epochs": 4, "learning_rate": 7.5e-5, "weight_decay": 1e-4,
        "max_steps_per_epoch": 10, "lambda_forget": 1.0, "lambda_retain": 2.0,
        "min_epochs": 2, "early_stop_forget_success": 1.0,
        "blindspot_temperature": 2.0, "blindspot_unknown_confidence": 0.90,
    },
    "bnd": {
        "epochs": 5, "learning_rate": 7.5e-5, "weight_decay": 1e-4,
        "max_steps_per_epoch": 10, "lambda_forget": 1.0, "lambda_retain": 2.0,
        "min_epochs": 2, "early_stop_forget_success": 1.0,
        "boundary_margin": 0.5,
    },
    "mkmmd": {
        "epochs": 4, "learning_rate": 2.5e-5, "weight_decay": 1e-4,
        "max_steps_per_epoch": 10, "lambda_forget": 1.0, "lambda_retain": 2.5,
        "min_epochs": 2, "early_stop_forget_success": 1.0,
        "lambda_mkmmd_forget": 0.5, "lambda_mkmmd_retain": 2.0,
        "mkmmd_bandwidths": [0.5, 1.0, 2.0, 4.0],
    },
}
TRAINABLE_SCOPES = ("only_head", "full_mlp", "last_encoder_block", "full_model")


@dataclass(frozen=True)
class CsvRecord:
    instance_id: str
    dataset_id: str
    csv_path: str
    byte_offset: int
    line_number: int
    row_sha256: str
    source: str
    original_label: str
    original_label_id: int
    binary_label: int
    split_file: str


@dataclass(frozen=True)
class PgdConfig:
    steps: int
    epsilon_time: float
    epsilon_size: float
    alpha_time: float
    alpha_size: float
    max_packet_size: float
    round_packet_size: bool


@dataclass(frozen=True)
class ModeHyperparameters:
    epochs: int
    learning_rate: float
    weight_decay: float
    max_steps_per_epoch: int
    lambda_forget: float
    lambda_retain: float
    min_epochs: int
    early_stop_forget_success: float
    blindspot_temperature: float = 2.0
    blindspot_unknown_confidence: float = 0.90
    boundary_margin: float = 0.5
    lambda_mkmmd_forget: float = 0.0
    lambda_mkmmd_retain: float = 0.0
    mkmmd_bandwidths: tuple[float, ...] = (0.5, 1.0, 2.0, 4.0)


def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    if hasattr(torch.backends, "cudnn"):
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False


def write_json(path: Path, value: object) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(value, ensure_ascii=False, indent=2), encoding="utf-8")


def write_csv_rows(path: Path, rows: Sequence[dict[str, object]]) -> None:
    if not rows:
        return
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", newline="", encoding="utf-8") as handle:
        writer = csv.DictWriter(handle, fieldnames=list(rows[0]))
        writer.writeheader()
        writer.writerows(rows)


def sha256_file(path: Path, chunk_size: int = 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        while chunk := handle.read(chunk_size):
            digest.update(chunk)
    return digest.hexdigest()


def state_to_cpu(state: dict[str, Tensor]) -> dict[str, Tensor]:
    return {name: value.detach().cpu() for name, value in state.items()}


def torch_load(path: Path, map_location: str | torch.device = "cpu") -> dict[str, Any]:
    try:
        value = torch.load(path, map_location=map_location, weights_only=True)
    except TypeError:
        value = torch.load(path, map_location=map_location)
    if not isinstance(value, dict):
        raise TypeError(f"Checkpoint phải là dictionary: {path}")
    return value


def parse_csv_values(value: str) -> tuple[str, ...]:
    return tuple(item.strip() for item in value.split(",") if item.strip())


def parse_int_values(value: str) -> tuple[int, ...]:
    result = tuple(int(item) for item in parse_csv_values(value))
    if not result:
        raise ValueError("Danh sách số nguyên không được rỗng")
    return result


def parse_float_values(value: str) -> tuple[float, ...]:
    result = tuple(float(item) for item in parse_csv_values(value))
    if not result:
        raise ValueError("Danh sách số thực không được rỗng")
    return result


def parse_mode_hyperparameters(
    value: str, modes: Sequence[str]
) -> dict[str, ModeHyperparameters]:
    payload = json.loads(value)
    if not isinstance(payload, dict):
        raise ValueError("mode_config_json must encode an object")
    unknown_modes = set(payload) - set(MODE_SPECS)
    if unknown_modes:
        raise ValueError(f"Unknown modes in mode config: {sorted(unknown_modes)}")
    missing_modes = set(modes) - set(payload)
    if missing_modes:
        raise ValueError(f"Missing mode configs: {sorted(missing_modes)}")

    configs: dict[str, ModeHyperparameters] = {}
    for mode in modes:
        raw = payload[mode]
        if not isinstance(raw, dict):
            raise ValueError(f"Mode config must be an object: {mode}")
        config = ModeHyperparameters(
            epochs=int(raw["epochs"]),
            learning_rate=float(raw["learning_rate"]),
            weight_decay=float(raw["weight_decay"]),
            max_steps_per_epoch=int(raw["max_steps_per_epoch"]),
            lambda_forget=float(raw["lambda_forget"]),
            lambda_retain=float(raw["lambda_retain"]),
            min_epochs=int(raw["min_epochs"]),
            early_stop_forget_success=float(raw["early_stop_forget_success"]),
            blindspot_temperature=float(raw.get("blindspot_temperature", 2.0)),
            blindspot_unknown_confidence=float(
                raw.get("blindspot_unknown_confidence", 0.90)
            ),
            boundary_margin=float(raw.get("boundary_margin", 0.5)),
            lambda_mkmmd_forget=float(raw.get("lambda_mkmmd_forget", 0.0)),
            lambda_mkmmd_retain=float(raw.get("lambda_mkmmd_retain", 0.0)),
            mkmmd_bandwidths=tuple(
                float(item)
                for item in raw.get("mkmmd_bandwidths", (0.5, 1.0, 2.0, 4.0))
            ),
        )
        if mode == "baseline":
            if config.epochs != 0:
                raise ValueError("baseline epochs must be 0")
        else:
            if config.epochs < 1 or not 1 <= config.min_epochs <= config.epochs:
                raise ValueError(f"Invalid epoch budget for {mode}: {config}")
            if config.learning_rate <= 0 or config.weight_decay < 0:
                raise ValueError(f"Invalid optimizer config for {mode}: {config}")
            if config.max_steps_per_epoch < 0:
                raise ValueError(f"max_steps_per_epoch must be >= 0 for {mode}")
            if config.lambda_forget <= 0 or config.lambda_retain < 0:
                raise ValueError(f"Invalid loss weights for {mode}: {config}")
            if not 0 < config.early_stop_forget_success <= 1:
                raise ValueError(f"Invalid early-stop target for {mode}: {config}")
        if config.blindspot_temperature <= 0:
            raise ValueError(f"blindspot_temperature must be > 0 for {mode}")
        if not 0.5 < config.blindspot_unknown_confidence < 1.0:
            raise ValueError(f"Invalid blindspot confidence for {mode}")
        if config.boundary_margin <= 0:
            raise ValueError(f"boundary_margin must be > 0 for {mode}")
        if config.lambda_mkmmd_forget < 0 or config.lambda_mkmmd_retain < 0:
            raise ValueError(f"Invalid MKMMD weights for {mode}")
        if not config.mkmmd_bandwidths or any(
            bandwidth <= 0 for bandwidth in config.mkmmd_bandwidths
        ):
            raise ValueError(f"Invalid MKMMD bandwidths for {mode}")
        configs[mode] = config
    return configs


## 2. CSV paths, byte-offset index, stable instance IDs, and splits


In [ ]:
def file_signature(path: Path) -> dict[str, object]:
    stat = path.stat()
    return {
        "path": str(path.resolve()),
        "bytes": stat.st_size,
        "mtime_ns": stat.st_mtime_ns,
    }


def dataset_signatures(aol_dir: Path, unknown_dir: Path) -> list[dict[str, object]]:
    paths = [
        root / name
        for root in (aol_dir, unknown_dir)
        for name in ("train_data.csv", "test_data.csv", "label_mapping.csv")
    ]
    return [file_signature(path) for path in paths]


def preflight_dataset_dirs(aol_dir: Path, unknown_dir: Path) -> None:
    if aol_dir.resolve() == unknown_dir.resolve():
        raise ValueError("AOL_DATASET_DIR và UNKNOWN_100_DATASET_DIR không được trùng nhau")
    for root in (aol_dir, unknown_dir):
        if not root.is_dir():
            raise FileNotFoundError(f"Không tìm thấy dataset directory: {root}")
        for name in ("train_data.csv", "test_data.csv", "label_mapping.csv"):
            path = root / name
            if not path.is_file() or path.stat().st_size == 0:
                raise FileNotFoundError(f"Thiếu hoặc rỗng: {path}")


def read_label_mapping(path: Path) -> dict[int, str]:
    with path.open(newline="", encoding="utf-8") as handle:
        reader = csv.DictReader(handle)
        if reader.fieldnames != ["label_name", "label_id"]:
            raise ValueError(
                f"label_mapping.csv phải có header label_name,label_id; nhận {reader.fieldnames}"
            )
        mapping: dict[int, str] = {}
        for row in reader:
            label_id = int(row["label_id"])
            label_name = row["label_name"] or f"label_{label_id}"
            if label_id in mapping:
                raise ValueError(f"Trùng label_id={label_id} trong {path}")
            mapping[label_id] = label_name
    if not mapping:
        raise ValueError(f"Không có label trong {path}")
    return mapping


def make_instance_id(dataset_id: str, source: str, split_file: str, line_number: int) -> str:
    return f"{dataset_id}:{source}:{split_file}:{line_number}"


def index_one_csv(
    csv_path: Path,
    mapping: dict[int, str],
    dataset_id: str,
    source: str,
    binary_label: int,
    split_file: str,
) -> list[CsvRecord]:
    records: list[CsvRecord] = []
    with csv_path.open("rb") as handle:
        line_number = 0
        while True:
            byte_offset = handle.tell()
            raw = handle.readline()
            if not raw:
                break
            line_number += 1
            comma = raw.find(b",")
            if comma < 0:
                raise ValueError(f"Dòng không có dấu phẩy: {csv_path}:{line_number}")
            try:
                original_label_id = int(float(raw[:comma]))
            except ValueError as error:
                raise ValueError(f"Label không hợp lệ: {csv_path}:{line_number}") from error
            if original_label_id not in mapping:
                raise ValueError(
                    f"label_id={original_label_id} không có trong mapping: {csv_path}:{line_number}"
                )
            records.append(
                CsvRecord(
                    instance_id=make_instance_id(dataset_id, source, split_file, line_number),
                    dataset_id=dataset_id,
                    csv_path=str(csv_path.resolve()),
                    byte_offset=byte_offset,
                    line_number=line_number,
                    row_sha256=hashlib.sha256(raw.rstrip(b"\r\n")).hexdigest(),
                    source=source,
                    original_label=mapping[original_label_id],
                    original_label_id=original_label_id,
                    binary_label=binary_label,
                    split_file=split_file,
                )
            )
    if not records:
        raise ValueError(f"CSV không có instance: {csv_path}")
    return records


def load_or_build_index(
    aol_dir: Path,
    unknown_dir: Path,
    index_path: Path,
) -> tuple[list[CsvRecord], list[CsvRecord], list[dict[str, object]]]:
    signatures = dataset_signatures(aol_dir, unknown_dir)
    if index_path.is_file():
        cached = json.loads(index_path.read_text(encoding="utf-8"))
        if (
            cached.get("version") == INDEX_VERSION
            and cached.get("signatures") == signatures
        ):
            return (
                [CsvRecord(**row) for row in cached["source_train"]],
                [CsvRecord(**row) for row in cached["test"]],
                signatures,
            )

    aol_mapping = read_label_mapping(aol_dir / "label_mapping.csv")
    unknown_mapping = read_label_mapping(unknown_dir / "label_mapping.csv")
    source_train = index_one_csv(
        aol_dir / "train_data.csv", aol_mapping, "aol_v1", "AOL", KNOWN, "train_data.csv"
    ) + index_one_csv(
        unknown_dir / "train_data.csv",
        unknown_mapping,
        "icloud100_v1",
        "100_TOPSEARCH",
        UNKNOWN,
        "train_data.csv",
    )
    test = index_one_csv(
        aol_dir / "test_data.csv", aol_mapping, "aol_v1", "AOL", KNOWN, "test_data.csv"
    ) + index_one_csv(
        unknown_dir / "test_data.csv",
        unknown_mapping,
        "icloud100_v1",
        "100_TOPSEARCH",
        UNKNOWN,
        "test_data.csv",
    )
    write_json(
        index_path,
        {
            "version": INDEX_VERSION,
            "signatures": signatures,
            "source_train": [asdict(row) for row in source_train],
            "test": [asdict(row) for row in test],
        },
    )
    return source_train, test, signatures


def split_records(
    source_train: Sequence[CsvRecord],
    test: Sequence[CsvRecord],
    seed: int,
    valid_ratio: float,
) -> tuple[list[CsvRecord], list[CsvRecord], list[CsvRecord]]:
    groups: dict[tuple[str, str], list[CsvRecord]] = defaultdict(list)
    for record in source_train:
        groups[(record.source, record.original_label)].append(record)
    train: list[CsvRecord] = []
    validation: list[CsvRecord] = []
    rng = random.Random(seed)
    for key in sorted(groups):
        items = list(groups[key])
        rng.shuffle(items)
        n_val = max(1, round(len(items) * valid_ratio)) if len(items) >= 3 else 0
        validation.extend(items[:n_val])
        train.extend(items[n_val:])
    rng.shuffle(train)
    rng.shuffle(validation)
    result_test = list(test)
    if not train or not validation or not result_test:
        raise ValueError("Train/validation/test không được rỗng")
    if set(row.instance_id for row in train) & set(row.instance_id for row in validation):
        raise AssertionError("Train và validation bị giao nhau")
    return train, validation, result_test


## 3. Row decoding, tensor cache, and data loaders


In [ ]:
def cache_path(cache_dir: Path, record: CsvRecord) -> Path:
    key = hashlib.sha1(
        f"{FEATURE_TRANSFORM}:{ROW_WIDTH}:{record.instance_id}:{record.row_sha256}".encode()
    ).hexdigest()
    return cache_dir / f"{key}.pt"


def decode_csv_record(record: CsvRecord) -> tuple[Tensor, Tensor]:
    with Path(record.csv_path).open("rb") as handle:
        handle.seek(record.byte_offset)
        raw = handle.readline().rstrip(b"\r\n")
    if hashlib.sha256(raw).hexdigest() != record.row_sha256:
        raise ValueError(f"Row hash thay đổi: {record.instance_id}")
    values = np.fromstring(raw.decode("ascii"), sep=",", dtype=np.float32)
    if values.size != ROW_WIDTH:
        raise ValueError(
            f"Cần {ROW_WIDTH} values, nhận {values.size}: {record.instance_id}"
        )
    packets = values.reshape(MAX_PACKETS, RAW_FIELDS_PER_PACKET)
    labels = packets[:, 0]
    features_np = packets[:, 1:]
    real_mask = features_np[:, 2] != 0
    if not np.allclose(labels[real_mask], float(record.original_label_id)):
        raise ValueError(f"Label packet không đồng nhất: {record.instance_id}")
    padding = ~real_mask
    if padding.any() and not np.allclose(packets[padding], 0.0):
        raise ValueError(f"Padding phải là [0,0,0,0]: {record.instance_id}")
    features = torch.from_numpy(features_np.copy())
    mask = torch.from_numpy(real_mask.copy())
    return features, mask


def load_features(record: CsvRecord, feature_cache_dir: Path | None) -> tuple[Tensor, Tensor]:
    if feature_cache_dir is None:
        return decode_csv_record(record)
    feature_cache_dir.mkdir(parents=True, exist_ok=True)
    path = cache_path(feature_cache_dir, record)
    if path.is_file():
        payload = torch_load(path)
        return payload["features"], payload["mask"]
    features, mask = decode_csv_record(record)
    temporary = path.with_suffix(".tmp")
    torch.save({"features": features, "mask": mask}, temporary)
    temporary.replace(path)
    return features, mask


class CsvFlowDataset(Dataset[tuple[Tensor, Tensor, Tensor, str]]):
    def __init__(self, records: Sequence[CsvRecord], feature_cache_dir: Path | None):
        self.records = list(records)
        self.feature_cache_dir = feature_cache_dir
        self.labels = torch.tensor([row.binary_label for row in records], dtype=torch.long)

    def __len__(self) -> int:
        return len(self.records)

    def __getitem__(self, index: int) -> tuple[Tensor, Tensor, Tensor, str]:
        record = self.records[index]
        features, mask = load_features(record, self.feature_cache_dir)
        return features, mask, self.labels[index], record.instance_id


class AdversarialDataset(Dataset[tuple[Tensor, Tensor, Tensor, str]]):
    def __init__(self, payload: dict[str, Any]):
        self.features: Tensor = payload["features"]
        self.masks: Tensor = payload["masks"]
        self.targets: Tensor = payload["targets"]
        self.instance_ids: list[str] = list(payload["instance_ids"])

    def __len__(self) -> int:
        return self.features.shape[0]

    def __getitem__(self, index: int) -> tuple[Tensor, Tensor, Tensor, str]:
        return (
            self.features[index],
            self.masks[index],
            self.targets[index],
            self.instance_ids[index],
        )


def make_loader(
    records: Sequence[CsvRecord],
    batch_size: int,
    shuffle: bool,
    num_workers: int,
    device: torch.device,
    feature_cache_dir: Path | None,
) -> DataLoader:
    return DataLoader(
        CsvFlowDataset(records, feature_cache_dir),
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=num_workers,
        pin_memory=device.type == "cuda",
        persistent_workers=num_workers > 0,
    )


def make_adversarial_loader(
    payload: dict[str, Any], batch_size: int, shuffle: bool, device: torch.device
) -> DataLoader:
    return DataLoader(
        AdversarialDataset(payload),
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=0,
        pin_memory=device.type == "cuda",
    )


def move_batch(batch: Sequence[Any], device: torch.device) -> tuple[Tensor, Tensor, Tensor]:
    features, mask, labels = batch[:3]
    return (
        features.to(device, non_blocking=True),
        mask.to(device, non_blocking=True),
        labels.to(device, non_blocking=True),
    )


def endless_batches(loader: DataLoader) -> Iterator[Sequence[Any]]:
    """Repeat a loader without caching every decoded batch in host memory."""
    while True:
        yield from loader


## 4. Checkpoint-compatible encoder and binary MLP


In [ ]:
class FlowEncoder(nn.Module):
    """Checkpoint-compatible legacy raw-packet CNN encoder."""

    def __init__(self, max_packets: int = MAX_PACKETS, embedding_dim: int = EMBEDDING_DIM):
        super().__init__()
        self.max_packets = max_packets
        self.embedding_dim = embedding_dim

        def block(cin: int, cout: int, suffix: int) -> None:
            setattr(self, f"conv{suffix}", nn.Conv1d(cin, cout, 8))
            setattr(self, f"conv{suffix}_{suffix}", nn.Conv1d(cout, cout, 8))
            setattr(self, f"batch_norm{suffix}", nn.BatchNorm1d(cout))
            setattr(self, f"max_pool_{suffix}", nn.MaxPool1d(8, stride=4))
            setattr(self, f"dropout{suffix}", nn.Dropout(0.1))

        block(3, 32, 1)
        block(32, 64, 2)
        block(64, 128, 3)
        block(128, 256, 4)
        with torch.no_grad():
            flat = self._convs(torch.zeros(1, 3, max_packets)).flatten(1).shape[1]
        self.fc = nn.Linear(flat, embedding_dim)

    @staticmethod
    def _same_pad(x: Tensor) -> Tensor:
        return F.pad(x, (3, 4))

    def _one(self, x: Tensor, index: int, first: bool = False) -> Tensor:
        x = self._same_pad(x)
        x = (
            F.elu(getattr(self, f"conv{index}")(x))
            if first
            else F.relu(getattr(self, f"conv{index}")(x))
        )
        x = self._same_pad(x)
        x = getattr(self, f"conv{index}_{index}")(x)
        x = getattr(self, f"batch_norm{index}")(x)
        x = F.elu(x) if first else F.relu(x)
        x = getattr(self, f"max_pool_{index}")(self._same_pad(x))
        return getattr(self, f"dropout{index}")(x)

    def _convs(self, x: Tensor) -> Tensor:
        for index in range(1, 5):
            x = self._one(x, index, first=index == 1)
        return x

    def forward(self, features: Tensor, mask: Tensor | None = None) -> Tensor:
        if mask is not None:
            features = features * mask.unsqueeze(-1)
        return self.fc(self._convs(features.transpose(1, 2)).flatten(1))


class FlowModel(nn.Module):
    def __init__(
        self,
        hidden_dims: Sequence[int] = (512, 256, 128, 64, 32),
        dropout: float = 0.2,
    ):
        super().__init__()
        self.encoder = FlowEncoder()
        layers: list[nn.Module] = []
        width = EMBEDDING_DIM
        for next_width in hidden_dims:
            layers.extend(
                [
                    nn.Linear(width, next_width),
                    nn.LayerNorm(next_width),
                    nn.GELU(),
                    nn.Dropout(dropout),
                ]
            )
            width = next_width
        layers.append(nn.Linear(width, NUM_CLASSES))
        self.classifier = nn.Sequential(*layers)
        self.hidden_dims = tuple(hidden_dims)
        self.dropout = dropout

    def forward(self, features: Tensor, mask: Tensor | None = None) -> tuple[Tensor, Tensor]:
        embedding = self.encoder(features, mask)
        return self.classifier(embedding), embedding


def load_pretrained_encoder(model: FlowModel, checkpoint_path: Path) -> dict[str, Any]:
    checkpoint = torch_load(checkpoint_path)
    config = checkpoint.get("model_config", {})
    required = {"max_packets": MAX_PACKETS, "hidden_size": 256, "embedding_size": 128}
    mismatch = {key: (config.get(key), expected) for key, expected in required.items() if config.get(key) != expected}
    if mismatch:
        raise ValueError(f"Checkpoint encoder không tương thích: {mismatch}")
    source_state = checkpoint.get("model_state_dict")
    if not isinstance(source_state, dict):
        raise ValueError(f"Thiếu model_state_dict trong {checkpoint_path}")
    encoder_state = {
        name.removeprefix("encoder."): value
        for name, value in source_state.items()
        if name.startswith("encoder.")
    }
    model.encoder.load_state_dict(encoder_state, strict=True)
    return {
        "path": str(checkpoint_path.resolve()),
        "sha256": sha256_file(checkpoint_path),
        "epoch": checkpoint.get("epoch"),
        "model_config": config,
    }


def resolve_device(device_name: str, allow_cpu: bool) -> torch.device:
    if device_name:
        requested = torch.device(device_name)
    else:
        requested = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
    if requested.type == "cuda" and not torch.cuda.is_available():
        if not allow_cpu:
            raise RuntimeError("CUDA không khả dụng; kiểm tra GPU VM hoặc thêm --allow-cpu để debug")
        return torch.device("cpu")
    return requested


## 5. Evaluation, base training, threshold selection, and checkpoint loading


In [ ]:
def evaluate(
    model: FlowModel,
    loader: DataLoader,
    device: torch.device,
    threshold: float,
) -> dict[str, object]:
    if len(loader.dataset) == 0:
        return {
            "loss": float("nan"),
            "accuracy": float("nan"),
            "known_recall": float("nan"),
            "unknown_recall": float("nan"),
            "balanced_accuracy": float("nan"),
            "unknown_threshold": threshold,
            "total": 0,
            "confusion_matrix": [[0, 0], [0, 0]],
        }
    model.eval()
    criterion = nn.CrossEntropyLoss()
    total = correct = known_total = known_ok = unknown_total = unknown_ok = 0
    loss_sum = 0.0
    confusion = torch.zeros(NUM_CLASSES, NUM_CLASSES, dtype=torch.long)
    with torch.no_grad():
        for batch in loader:
            features, mask, labels = move_batch(batch, device)
            logits, _ = model(features, mask)
            loss_sum += criterion(logits, labels).item() * labels.numel()
            predicted = (torch.softmax(logits, dim=1)[:, UNKNOWN] >= threshold).long()
            total += labels.numel()
            correct += (predicted == labels).sum().item()
            known = labels == KNOWN
            unknown = labels == UNKNOWN
            known_total += known.sum().item()
            unknown_total += unknown.sum().item()
            known_ok += ((predicted == labels) & known).sum().item()
            unknown_ok += ((predicted == labels) & unknown).sum().item()
            for true_label, predicted_label in zip(labels.cpu(), predicted.cpu()):
                confusion[int(true_label), int(predicted_label)] += 1
    known_recall = known_ok / known_total if known_total else float("nan")
    unknown_recall = unknown_ok / unknown_total if unknown_total else float("nan")
    valid_recalls = [value for value in (known_recall, unknown_recall) if not math.isnan(value)]
    balanced = sum(valid_recalls) / len(valid_recalls) if valid_recalls else float("nan")
    return {
        "loss": loss_sum / max(total, 1),
        "accuracy": correct / max(total, 1),
        "known_recall": known_recall,
        "unknown_recall": unknown_recall,
        "balanced_accuracy": balanced,
        "unknown_threshold": threshold,
        "total": total,
        "confusion_matrix": confusion.tolist(),
    }


def score_records(
    model: FlowModel, loader: DataLoader, device: torch.device, threshold: float
) -> list[dict[str, object]]:
    model.eval()
    rows: list[dict[str, object]] = []
    with torch.no_grad():
        for batch in loader:
            features, mask, labels = move_batch(batch, device)
            instance_ids = list(batch[3])
            logits, _ = model(features, mask)
            probability_unknown = torch.softmax(logits, dim=1)[:, UNKNOWN]
            predicted = (probability_unknown >= threshold).long()
            for instance_id, label, prediction, probability in zip(
                instance_ids, labels.cpu(), predicted.cpu(), probability_unknown.cpu()
            ):
                rows.append(
                    {
                        "instance_id": instance_id,
                        "label": int(label),
                        "prediction": int(prediction),
                        "probability_unknown": float(probability),
                    }
                )
    return rows


def forget_metrics(
    model: FlowModel, loader: DataLoader, device: torch.device, threshold: float
) -> dict[str, float]:
    rows = score_records(model, loader, device, threshold)
    if not rows:
        return {
            "forget_success_rate": float("nan"),
            "mean_probability_unknown": float("nan"),
            "count": 0.0,
        }
    return {
        "forget_success_rate": sum(row["prediction"] == UNKNOWN for row in rows) / len(rows),
        "mean_probability_unknown": sum(float(row["probability_unknown"]) for row in rows) / len(rows),
        "count": float(len(rows)),
    }


def class_weights(records: Sequence[CsvRecord], device: torch.device) -> Tensor:
    counts = torch.bincount(
        torch.tensor([record.binary_label for record in records]), minlength=NUM_CLASSES
    ).float()
    if (counts == 0).any():
        raise ValueError(f"Base train thiếu binary class: counts={counts.tolist()}")
    weights = counts.sum() / (NUM_CLASSES * counts)
    return (weights / weights.mean()).to(device)


def train_base_model(
    model: FlowModel,
    train_loader: DataLoader,
    validation_loader: DataLoader,
    train_records: Sequence[CsvRecord],
    device: torch.device,
    epochs: int,
    learning_rate: float,
    weight_decay: float,
    freeze_encoder: bool,
) -> tuple[dict[str, Tensor], list[dict[str, float]]]:
    if freeze_encoder:
        for parameter in model.encoder.parameters():
            parameter.requires_grad = False
    optimizer = torch.optim.AdamW(
        [parameter for parameter in model.parameters() if parameter.requires_grad],
        lr=learning_rate,
        weight_decay=weight_decay,
    )
    criterion = nn.CrossEntropyLoss(weight=class_weights(train_records, device))
    best_state = copy.deepcopy(model.state_dict())
    best_score = -float("inf")
    history: list[dict[str, float]] = []
    for epoch in range(1, epochs + 1):
        model.train()
        if freeze_encoder:
            model.encoder.eval()
        total_loss = 0.0
        total = 0
        for batch in train_loader:
            features, mask, labels = move_batch(batch, device)
            optimizer.zero_grad(set_to_none=True)
            logits, _ = model(features, mask)
            loss = criterion(logits, labels)
            loss.backward()
            nn.utils.clip_grad_norm_(
                [parameter for parameter in model.parameters() if parameter.requires_grad], 5.0
            )
            optimizer.step()
            total_loss += loss.item() * labels.numel()
            total += labels.numel()
        metrics = evaluate(model, validation_loader, device, threshold=0.5)
        score = float(metrics["balanced_accuracy"])
        if score > best_score:
            best_score = score
            best_state = copy.deepcopy(model.state_dict())
        row = {
            "epoch": float(epoch),
            "train_loss": total_loss / max(total, 1),
            "validation_accuracy": float(metrics["accuracy"]),
            "validation_balanced_accuracy": score,
            "validation_known_recall": float(metrics["known_recall"]),
            "validation_unknown_recall": float(metrics["unknown_recall"]),
        }
        history.append(row)
        print(
            f"Base epoch {epoch}/{epochs}: loss={row['train_loss']:.4f} "
            f"val_bal_acc={score:.4f}"
        )
    return best_state, history


def select_threshold(
    model: FlowModel,
    validation_loader: DataLoader,
    device: torch.device,
    thresholds: Sequence[float],
) -> tuple[float, list[dict[str, float]]]:
    rows: list[dict[str, float]] = []
    for threshold in thresholds:
        metrics = evaluate(model, validation_loader, device, threshold)
        rows.append(
            {
                "threshold": float(threshold),
                "accuracy": float(metrics["accuracy"]),
                "balanced_accuracy": float(metrics["balanced_accuracy"]),
                "known_recall": float(metrics["known_recall"]),
                "unknown_recall": float(metrics["unknown_recall"]),
            }
        )
    best = max(
        rows,
        key=lambda row: (
            row["balanced_accuracy"],
            row["unknown_recall"],
            row["accuracy"],
        ),
    )
    return best["threshold"], rows


def records_by_ids(
    records: Sequence[CsvRecord], ids: Sequence[str], label: str
) -> list[CsvRecord]:
    mapping = {record.instance_id: record for record in records}
    missing = [instance_id for instance_id in ids if instance_id not in mapping]
    if missing:
        raise ValueError(f"{label} có instance không tồn tại: {missing[:5]}")
    return [mapping[instance_id] for instance_id in ids]


def split_ids_payload(
    train: Sequence[CsvRecord], validation: Sequence[CsvRecord], test: Sequence[CsvRecord]
) -> dict[str, list[str]]:
    return {
        "train": [row.instance_id for row in train],
        "validation": [row.instance_id for row in validation],
        "test": [row.instance_id for row in test],
    }


def load_base_context(
    args: argparse.Namespace,
    device: torch.device,
) -> tuple[
    FlowModel,
    dict[str, Any],
    list[CsvRecord],
    list[CsvRecord],
    list[CsvRecord],
    list[dict[str, object]],
]:
    checkpoint_path = Path(args.base_checkpoint).resolve()
    checkpoint = torch_load(checkpoint_path, map_location=device)
    model_config = checkpoint.get("model_config", {})
    model = FlowModel(
        hidden_dims=tuple(model_config.get("hidden_dims", (512, 256, 128, 64, 32))),
        dropout=float(model_config.get("dropout", 0.2)),
    ).to(device)
    model.load_state_dict(checkpoint["model_state_dict"], strict=True)

    aol_dir = Path(args.aol_dir).resolve()
    unknown_dir = Path(args.unknown_dir).resolve()
    preflight_dataset_dirs(aol_dir, unknown_dir)
    index_path = Path(args.index_path).resolve()
    source_train, source_test, signatures = load_or_build_index(
        aol_dir, unknown_dir, index_path
    )
    all_records = source_train + source_test
    split_ids = checkpoint.get("split_ids")
    if split_ids:
        train = records_by_ids(all_records, split_ids["train"], "base train")
        validation = records_by_ids(all_records, split_ids["validation"], "base validation")
        test = records_by_ids(all_records, split_ids["test"], "base test")
    else:
        print("Cảnh báo: base checkpoint cũ không có split_ids; tái tạo split deterministic.")
        train, validation, test = split_records(
            source_train, source_test, args.seed, args.valid_ratio
        )
    expected_signatures = checkpoint.get("dataset_signatures")
    if expected_signatures and expected_signatures != signatures:
        raise ValueError("Dataset signatures không khớp base checkpoint")
    checkpoint["checkpoint_path"] = str(checkpoint_path)
    checkpoint["checkpoint_sha256"] = sha256_file(checkpoint_path)
    return model, checkpoint, train, validation, test, signatures


## 6. Independent benchmark modes and adapted paper methods

The benchmark uses one flat mode list, not a `method x scope` Cartesian product.
`only_head`, `full_mlp`, `last_encoder_block`, and `full_model` are independent
targeted-rehearsal baselines. Each paper method is another independent mode with its
own fixed trainable scope and loss helper. All modes share data and evaluation code.
`Amnesiac` is adapted to post-hoc gradient ascent because the base run does not store
per-batch updates. `Blindspot` uses the base model as the competent teacher and a soft
unknown target as the task-specific incompetent teacher. `BND` applies a binary logit
margin. `MKMMD` aligns forget embeddings with unknown-reference embeddings while
anchoring retain embeddings to the frozen base teacher. These adaptations are recorded
in every result artifact; they are not claimed as exact reproductions of the papers.


In [ ]:
def set_scope(model: FlowModel, scope: str) -> list[str]:
    if scope not in TRAINABLE_SCOPES:
        raise ValueError(f"Scope không hợp lệ: {scope}")
    for parameter in model.parameters():
        parameter.requires_grad = False
    modules: dict[str, Sequence[nn.Module]] = {
        "only_head": (model.classifier[-1],),
        "full_mlp": (model.classifier,),
        "last_encoder_block": (
            model.encoder.conv4,
            model.encoder.conv4_4,
            model.encoder.batch_norm4,
            model.encoder.fc,
            model.classifier,
        ),
        "full_model": (model.encoder, model.classifier),
    }
    for module in modules[scope]:
        for parameter in module.parameters():
            parameter.requires_grad = True
    return [name for name, parameter in model.named_parameters() if parameter.requires_grad]


def set_unlearning_train_mode(model: FlowModel, scope: str) -> None:
    model.train()
    if scope == "only_head":
        model.eval()
        model.classifier[-1].train()
    elif scope == "full_mlp":
        model.encoder.eval()
        model.classifier.train()
    elif scope == "last_encoder_block":
        model.encoder.eval()
        for module in (
            model.encoder.conv4,
            model.encoder.conv4_4,
            model.encoder.batch_norm4,
            model.encoder.max_pool_4,
            model.encoder.dropout4,
            model.encoder.fc,
        ):
            module.train()
        model.classifier.train()


def estimate_mas_importance(
    model: FlowModel,
    forget_loader: DataLoader,
    device: torch.device,
    trainable_names: Sequence[str],
) -> dict[str, Tensor]:
    """Estimate MAS importance on Df using gradients of the output L2 norm."""
    model.eval()
    named = dict(model.named_parameters())
    parameters = [named[name] for name in trainable_names]
    importance = {name: torch.zeros_like(named[name]) for name in trainable_names}
    total = 0
    for batch in forget_loader:
        features, mask, _ = move_batch(batch, device)
        model.zero_grad(set_to_none=True)
        logits, _ = model(features, mask)
        output_norm = torch.norm(logits, p=2, dim=1).sum()
        gradients = torch.autograd.grad(output_norm, parameters, allow_unused=True)
        for name, gradient in zip(trainable_names, gradients):
            if gradient is not None:
                importance[name] += gradient.detach().abs()
        total += features.shape[0]
    if total == 0:
        raise ValueError("Df rỗng khi tính MAS")
    return {name: value / total for name, value in importance.items()}


def inverted_mas_anchors(importance: dict[str, Tensor]) -> dict[str, Tensor]:
    anchors: dict[str, Tensor] = {}
    for name, values in importance.items():
        minimum = values.min()
        span = values.max() - minimum
        if float(span) <= 1e-12:
            anchors[name] = torch.ones_like(values)
        else:
            anchors[name] = 1.0 - (values - minimum) / span
    return anchors


def parameter_anchor_loss(
    model: FlowModel,
    reference: dict[str, Tensor],
    anchors: dict[str, Tensor],
) -> Tensor:
    loss = torch.zeros((), device=next(model.parameters()).device)
    named = dict(model.named_parameters())
    for name, anchor in anchors.items():
        loss = loss + (anchor * (named[name] - reference[name]).pow(2)).sum() / 2
    return loss


def constrain_adversarial(
    candidate: Tensor,
    original: Tensor,
    mask: Tensor,
    config: PgdConfig,
) -> Tensor:
    real = mask.unsqueeze(-1)
    result = original.clone()

    time_delta = (candidate[:, :, 0] - original[:, :, 0]).clamp(
        -config.epsilon_time, config.epsilon_time
    )
    time_values = (original[:, :, 0] + time_delta).clamp_min(0.0)
    time_values = torch.cummax(time_values, dim=1).values
    time_values = torch.maximum(
        torch.minimum(time_values, original[:, :, 0] + config.epsilon_time),
        (original[:, :, 0] - config.epsilon_time).clamp_min(0.0),
    )
    # Preserve the timestamp origin of the first packet.
    time_values[:, 0] = original[:, 0, 0]

    size_delta = (candidate[:, :, 2] - original[:, :, 2]).clamp(
        -config.epsilon_size, config.epsilon_size
    )
    size_values = (original[:, :, 2] + size_delta).clamp(1.0, config.max_packet_size)
    if config.round_packet_size:
        size_values = size_values.round()

    result[:, :, 0] = torch.where(mask, time_values, torch.zeros_like(time_values))
    result[:, :, 1] = original[:, :, 1]
    result[:, :, 2] = torch.where(mask, size_values, torch.zeros_like(size_values))
    return torch.where(real, result, torch.zeros_like(result))


def generate_targeted_adversarial(
    base_model: FlowModel,
    forget_loader: DataLoader,
    device: torch.device,
    config: PgdConfig,
) -> dict[str, Any]:
    """Generate targeted unknown examples while preserving traffic constraints."""
    attack_model = copy.deepcopy(base_model).to(device).eval()
    for parameter in attack_model.parameters():
        parameter.requires_grad = False
    all_features: list[Tensor] = []
    all_masks: list[Tensor] = []
    all_targets: list[Tensor] = []
    all_ids: list[str] = []
    for batch in forget_loader:
        original, mask, _ = move_batch(batch, device)
        instance_ids = list(batch[3])
        target = torch.full(
            (original.shape[0],), UNKNOWN, dtype=torch.long, device=device
        )
        adversarial = original.detach().clone()
        for _ in range(config.steps):
            adversarial.requires_grad_(True)
            logits, _ = attack_model(adversarial, mask)
            target_loss = F.cross_entropy(logits, target)
            gradient = torch.autograd.grad(target_loss, adversarial)[0]
            updated = adversarial.detach().clone()
            updated[:, :, 0] -= config.alpha_time * gradient[:, :, 0].sign()
            updated[:, :, 2] -= config.alpha_size * gradient[:, :, 2].sign()
            adversarial = constrain_adversarial(updated, original, mask, config).detach()
        all_features.append(adversarial.cpu())
        all_masks.append(mask.cpu())
        all_targets.append(target.cpu())
        all_ids.extend(instance_ids)
    return {
        "features": torch.cat(all_features),
        "masks": torch.cat(all_masks),
        "targets": torch.cat(all_targets),
        "instance_ids": all_ids,
        "config": asdict(config),
    }


def adversarial_cache_key(
    checkpoint_sha256: str, forget_records: Sequence[CsvRecord], config: PgdConfig
) -> str:
    value = {
        "checkpoint": checkpoint_sha256,
        "instances": [record.instance_id for record in forget_records],
        "row_hashes": [record.row_sha256 for record in forget_records],
        "pgd": asdict(config),
        "feature_transform": FEATURE_TRANSFORM,
    }
    return hashlib.sha256(json.dumps(value, sort_keys=True).encode()).hexdigest()[:20]


def load_or_generate_adversarial(
    base_model: FlowModel,
    forget_records: Sequence[CsvRecord],
    forget_loader: DataLoader,
    checkpoint_sha256: str,
    cache_dir: Path,
    device: torch.device,
    config: PgdConfig,
) -> tuple[dict[str, Any], Path]:
    cache_dir.mkdir(parents=True, exist_ok=True)
    path = cache_dir / f"targeted_unknown_{adversarial_cache_key(checkpoint_sha256, forget_records, config)}.pt"
    if path.is_file():
        return torch_load(path), path
    payload = generate_targeted_adversarial(base_model, forget_loader, device, config)
    torch.save(payload, path)
    return payload, path


def amnesiac_gradient_ascent_loss(logits: Tensor, original_labels: Tensor) -> Tensor:
    """Post-hoc adaptation used because base training did not store batch updates."""
    return -F.cross_entropy(logits, original_labels)


def distillation_kl(
    student_logits: Tensor, teacher_logits: Tensor, temperature: float
) -> Tensor:
    teacher_probability = F.softmax(teacher_logits / temperature, dim=1)
    return (
        F.kl_div(
            F.log_softmax(student_logits / temperature, dim=1),
            teacher_probability,
            reduction="batchmean",
        )
        * temperature**2
    )


def blindspot_unknown_teacher_loss(
    student_logits: Tensor, temperature: float, unknown_confidence: float
) -> Tensor:
    target = torch.empty_like(student_logits)
    target[:, KNOWN] = 1.0 - unknown_confidence
    target[:, UNKNOWN] = unknown_confidence
    return (
        F.kl_div(
            F.log_softmax(student_logits / temperature, dim=1),
            target,
            reduction="batchmean",
        )
        * temperature**2
    )


def boundary_margin_loss(logits: Tensor, margin: float) -> Tensor:
    unknown_margin = logits[:, UNKNOWN] - logits[:, KNOWN]
    return F.relu(margin - unknown_margin).mean()


def pairwise_squared_distance(left: Tensor, right: Tensor) -> Tensor:
    return (
        left.pow(2).sum(dim=1, keepdim=True)
        + right.pow(2).sum(dim=1).unsqueeze(0)
        - 2.0 * left @ right.transpose(0, 1)
    ).clamp_min(0.0)


def multi_kernel_mmd(
    left: Tensor, right: Tensor, bandwidth_multipliers: Sequence[float]
) -> Tensor:
    if left.ndim != 2 or right.ndim != 2 or left.shape[1] != right.shape[1]:
        raise ValueError("MKMMD expects two [batch, feature] tensors")
    if not bandwidth_multipliers or any(value <= 0 for value in bandwidth_multipliers):
        raise ValueError("MKMMD bandwidth multipliers must be positive")
    with torch.no_grad():
        combined = torch.cat((left.detach(), right.detach()), dim=0)
        distances = torch.pdist(combined, p=2).pow(2)
        positive = distances[distances > 0]
        base_bandwidth = positive.median() if positive.numel() else left.new_tensor(1.0)
        base_bandwidth = base_bandwidth.clamp_min(1e-6)

    distance_xx = pairwise_squared_distance(left, left)
    distance_yy = pairwise_squared_distance(right, right)
    distance_xy = pairwise_squared_distance(left, right)
    kernel_xx = torch.zeros_like(distance_xx)
    kernel_yy = torch.zeros_like(distance_yy)
    kernel_xy = torch.zeros_like(distance_xy)
    for multiplier in bandwidth_multipliers:
        denominator = 2.0 * base_bandwidth * float(multiplier)
        kernel_xx = kernel_xx + torch.exp(-distance_xx / denominator)
        kernel_yy = kernel_yy + torch.exp(-distance_yy / denominator)
        kernel_xy = kernel_xy + torch.exp(-distance_xy / denominator)
    scale = float(len(bandwidth_multipliers))
    return kernel_xx.mean() / scale + kernel_yy.mean() / scale - 2.0 * kernel_xy.mean() / scale


# Legacy L2UL helper retained for old artifacts; the benchmark command does not call it.
def _legacy_run_unlearning_method(
    base_model: FlowModel,
    method: str,
    scope: str,
    forget_loader: DataLoader,
    retain_loader: DataLoader,
    adversarial_payload: dict[str, Any] | None,
    validation_loaders: dict[str, DataLoader],
    device: torch.device,
    threshold: float,
    epochs: int,
    learning_rate: float,
    weight_decay: float,
    lambda_retain: float,
    lambda_adv: float,
    lambda_mas: float,
    batch_size: int,
    max_steps_per_epoch: int,
) -> tuple[FlowModel, list[dict[str, object]], dict[str, Tensor] | None]:
    if method not in LEGACY_METHODS:
        raise ValueError(f"Method không hợp lệ: {method}")
    branch = copy.deepcopy(base_model).to(device)
    trainable_names = set_scope(branch, scope)
    reference = {
        name: parameter.detach().clone()
        for name, parameter in branch.named_parameters()
        if name in trainable_names
    }
    anchors: dict[str, Tensor] | None = None
    if method in ("l2ul_mas", "l2ul_adv_mas"):
        importance = estimate_mas_importance(branch, forget_loader, device, trainable_names)
        anchors = inverted_mas_anchors(importance)

    adversarial_loader: DataLoader | None = None
    if method in ("l2ul_adv", "l2ul_adv_mas"):
        if adversarial_payload is None:
            raise ValueError(f"{method} yêu cầu adversarial payload")
        adversarial_loader = make_adversarial_loader(
            adversarial_payload, batch_size, shuffle=True, device=device
        )

    optimizer = torch.optim.AdamW(
        [parameter for parameter in branch.parameters() if parameter.requires_grad],
        lr=learning_rate,
        weight_decay=weight_decay,
    )
    criterion = nn.CrossEntropyLoss()
    forget_cycle = endless_batches(forget_loader)
    retain_cycle = endless_batches(retain_loader)
    adversarial_cycle = endless_batches(adversarial_loader) if adversarial_loader else None
    if method == "retain_rehearsal":
        default_steps = max(len(forget_loader), len(retain_loader))
    else:
        default_steps = len(forget_loader)
    steps = min(default_steps, max_steps_per_epoch) if max_steps_per_epoch > 0 else default_steps
    history: list[dict[str, object]] = []

    for epoch in range(1, epochs + 1):
        set_unlearning_train_mode(branch, scope)
        sums = {"total": 0.0, "forget": 0.0, "retain": 0.0, "adv": 0.0, "mas": 0.0}
        for _ in range(steps):
            forget_features, forget_mask, _ = move_batch(next(forget_cycle), device)
            forget_target = torch.full(
                (forget_features.shape[0],), UNKNOWN, dtype=torch.long, device=device
            )
            optimizer.zero_grad(set_to_none=True)
            forget_logits, _ = branch(forget_features, forget_mask)
            forget_loss = criterion(forget_logits, forget_target)
            loss = forget_loss
            retain_loss = torch.zeros((), device=device)
            adv_loss = torch.zeros((), device=device)
            mas_loss = torch.zeros((), device=device)

            if method == "retain_rehearsal":
                retain_features, retain_mask, retain_labels = move_batch(
                    next(retain_cycle), device
                )
                retain_logits, _ = branch(retain_features, retain_mask)
                retain_loss = criterion(retain_logits, retain_labels)
                loss = loss + lambda_retain * retain_loss

            if adversarial_cycle is not None:
                adv_features, adv_mask, adv_target = move_batch(
                    next(adversarial_cycle), device
                )
                adv_logits, _ = branch(adv_features, adv_mask)
                adv_loss = criterion(adv_logits, adv_target)
                loss = loss + lambda_adv * adv_loss

            if anchors is not None:
                mas_loss = parameter_anchor_loss(branch, reference, anchors)
                loss = loss + lambda_mas * mas_loss

            loss.backward()
            nn.utils.clip_grad_norm_(
                [parameter for parameter in branch.parameters() if parameter.requires_grad], 5.0
            )
            optimizer.step()
            sums["total"] += loss.item()
            sums["forget"] += forget_loss.item()
            sums["retain"] += retain_loss.item()
            sums["adv"] += adv_loss.item()
            sums["mas"] += mas_loss.item()

        row: dict[str, object] = {
            "epoch": epoch,
            "steps": steps,
            **{f"loss_{key}": value / max(steps, 1) for key, value in sums.items()},
            "forget": forget_metrics(branch, forget_loader, device, threshold),
            "retain_validation": {
                name: evaluate(branch, loader, device, threshold)
                for name, loader in validation_loaders.items()
            },
        }
        history.append(row)
        print(
            f"[{method}/{scope}] epoch {epoch}/{epochs} "
            f"loss={row['loss_total']:.5f} "
            f"forget={row['forget']['forget_success_rate']:.4f}"
        )
    return branch, history, anchors


def run_unlearning_mode(
    base_model: FlowModel,
    mode: str,
    forget_loader: DataLoader,
    retain_loader: DataLoader,
    unknown_reference_loader: DataLoader,
    validation_loaders: dict[str, DataLoader],
    device: torch.device,
    threshold: float,
    config: ModeHyperparameters,
) -> tuple[FlowModel, list[dict[str, object]]]:
    if mode not in MODE_SPECS:
        raise ValueError(f"Mode không hợp lệ: {mode}")
    spec = MODE_SPECS[mode]
    objective = str(spec["objective"])
    scope = str(spec["trainable_scope"])
    branch = copy.deepcopy(base_model).to(device)
    if objective == "baseline":
        branch.eval()
        return branch, []

    trainable_names = set_scope(branch, scope)
    trainable_parameters = [
        parameter for parameter in branch.parameters() if parameter.requires_grad
    ]
    if not trainable_parameters:
        raise ValueError(f"Mode {mode} không có trainable parameters")

    teacher: FlowModel | None = None
    if objective in ("blindspot", "mkmmd"):
        teacher = copy.deepcopy(base_model).to(device).eval()
        for parameter in teacher.parameters():
            parameter.requires_grad = False

    optimizer = torch.optim.AdamW(
        trainable_parameters,
        lr=config.learning_rate,
        weight_decay=config.weight_decay,
    )
    criterion = nn.CrossEntropyLoss()
    forget_cycle = endless_batches(forget_loader)
    retain_cycle = endless_batches(retain_loader)
    unknown_cycle = endless_batches(unknown_reference_loader)
    default_steps = max(len(forget_loader), min(len(retain_loader), 10), 1)
    steps = (
        min(default_steps, config.max_steps_per_epoch)
        if config.max_steps_per_epoch > 0
        else default_steps
    )
    history: list[dict[str, object]] = []

    for epoch in range(1, config.epochs + 1):
        set_unlearning_train_mode(branch, scope)
        sums = {
            "total": 0.0,
            "forget": 0.0,
            "retain": 0.0,
            "mkmmd_forget": 0.0,
            "mkmmd_retain": 0.0,
        }
        for _ in range(steps):
            forget_features, forget_mask, forget_labels = move_batch(
                next(forget_cycle), device
            )
            retain_features, retain_mask, retain_labels = move_batch(
                next(retain_cycle), device
            )
            forget_target = torch.full_like(forget_labels, UNKNOWN)
            optimizer.zero_grad(set_to_none=True)
            forget_logits, forget_embedding = branch(forget_features, forget_mask)
            retain_logits, retain_embedding = branch(retain_features, retain_mask)
            retain_loss = torch.zeros((), device=device)
            mkmmd_forget_loss = torch.zeros((), device=device)
            mkmmd_retain_loss = torch.zeros((), device=device)

            if objective == "targeted_rehearsal":
                forget_loss = criterion(forget_logits, forget_target)
                retain_loss = criterion(retain_logits, retain_labels)
                loss = (
                    config.lambda_forget * forget_loss
                    + config.lambda_retain * retain_loss
                )
            elif objective == "amnesiac":
                forget_loss = amnesiac_gradient_ascent_loss(
                    forget_logits, forget_labels
                )
                retain_loss = criterion(retain_logits, retain_labels)
                loss = (
                    config.lambda_forget * forget_loss
                    + config.lambda_retain * retain_loss
                )
            elif objective == "blindspot":
                if teacher is None:
                    raise AssertionError("Blindspot requires a frozen teacher")
                forget_loss = blindspot_unknown_teacher_loss(
                    forget_logits,
                    config.blindspot_temperature,
                    config.blindspot_unknown_confidence,
                )
                with torch.no_grad():
                    teacher_retain_logits, _ = teacher(retain_features, retain_mask)
                retain_loss = distillation_kl(
                    retain_logits,
                    teacher_retain_logits,
                    config.blindspot_temperature,
                )
                loss = (
                    config.lambda_forget * forget_loss
                    + config.lambda_retain * retain_loss
                )
            elif objective == "bnd":
                forget_loss = boundary_margin_loss(
                    forget_logits, config.boundary_margin
                )
                retain_loss = criterion(retain_logits, retain_labels)
                loss = (
                    config.lambda_forget * forget_loss
                    + config.lambda_retain * retain_loss
                )
            elif objective == "mkmmd":
                if teacher is None:
                    raise AssertionError("MKMMD requires a frozen teacher")
                unknown_features, unknown_mask, _ = move_batch(
                    next(unknown_cycle), device
                )
                _, unknown_embedding = branch(unknown_features, unknown_mask)
                with torch.no_grad():
                    _, teacher_retain_embedding = teacher(
                        retain_features, retain_mask
                    )
                forget_loss = criterion(forget_logits, forget_target)
                retain_loss = criterion(retain_logits, retain_labels)
                mkmmd_forget_loss = multi_kernel_mmd(
                    forget_embedding,
                    unknown_embedding,
                    config.mkmmd_bandwidths,
                )
                mkmmd_retain_loss = multi_kernel_mmd(
                    retain_embedding,
                    teacher_retain_embedding,
                    config.mkmmd_bandwidths,
                )
                loss = (
                    config.lambda_forget * forget_loss
                    + config.lambda_retain * retain_loss
                    + config.lambda_mkmmd_forget * mkmmd_forget_loss
                    + config.lambda_mkmmd_retain * mkmmd_retain_loss
                )
            else:
                raise ValueError(f"Objective chưa được triển khai: {objective}")

            loss.backward()
            nn.utils.clip_grad_norm_(trainable_parameters, 5.0)
            optimizer.step()
            sums["total"] += loss.item()
            sums["forget"] += forget_loss.item()
            sums["retain"] += retain_loss.item()
            sums["mkmmd_forget"] += mkmmd_forget_loss.item()
            sums["mkmmd_retain"] += mkmmd_retain_loss.item()

        row: dict[str, object] = {
            "epoch": epoch,
            "steps": steps,
            "mode": mode,
            "trainable_scope": scope,
            "trainable_parameter_names": trainable_names,
            **{f"loss_{key}": value / steps for key, value in sums.items()},
            "forget": forget_metrics(branch, forget_loader, device, threshold),
            "retain_validation": {
                name: evaluate(branch, loader, device, threshold)
                for name, loader in validation_loaders.items()
            },
        }
        history.append(row)
        print(
            f"[{mode}] epoch {epoch}/{config.epochs} "
            f"loss={row['loss_total']:.5f} "
            f"forget={row['forget']['forget_success_rate']:.4f}"
        )
        forget_success = float(row["forget"]["forget_success_rate"])
        if (
            epoch >= config.min_epochs
            and forget_success >= config.early_stop_forget_success
        ):
            print(
                f"[{mode}] early stop at epoch {epoch}: "
                f"forget_success={forget_success:.4f}"
            )
            break
    return branch, history


## 7. Forget manifests and retain evaluation slices


In [ ]:
def validate_forget_manifest(
    manifest_path: Path,
    train_records: Sequence[CsvRecord],
) -> tuple[dict[str, Any], list[CsvRecord]]:
    manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    entries = manifest.get("instances")
    if not isinstance(entries, list) or not entries:
        raise ValueError("Forget manifest phải có danh sách instances không rỗng")
    train_by_id = {record.instance_id: record for record in train_records}
    ids = [entry["instance_id"] for entry in entries]
    if len(ids) != len(set(ids)):
        raise ValueError("Forget manifest có instance_id trùng")
    selected: list[CsvRecord] = []
    for entry in entries:
        instance_id = entry["instance_id"]
        if instance_id not in train_by_id:
            raise ValueError(f"Instance không thuộc base train: {instance_id}")
        record = train_by_id[instance_id]
        if record.source != "AOL" or record.binary_label != KNOWN:
            raise ValueError(f"Chỉ được quên AOL known instance: {instance_id}")
        if entry.get("row_sha256") and entry["row_sha256"] != record.row_sha256:
            raise ValueError(f"Row hash không khớp: {instance_id}")
        selected.append(record)
    return manifest, selected


def stratified_nested_order(records: Sequence[CsvRecord], seed: int) -> list[CsvRecord]:
    groups: dict[str, list[CsvRecord]] = defaultdict(list)
    for record in records:
        groups[record.original_label].append(record)
    rng = random.Random(seed)
    for values in groups.values():
        rng.shuffle(values)
    label_order = sorted(groups)
    rng.shuffle(label_order)
    ordered: list[CsvRecord] = []
    while any(groups.values()):
        for label in label_order:
            if groups[label]:
                ordered.append(groups[label].pop())
    return ordered


def build_slice_loaders(
    records: Sequence[CsvRecord],
    forget_labels: set[str],
    args: argparse.Namespace,
    device: torch.device,
    feature_cache_dir: Path,
) -> dict[str, DataLoader]:
    slices = {
        "same_original_label": [
            row for row in records if row.source == "AOL" and row.original_label in forget_labels
        ],
        "other_known": [
            row for row in records if row.source == "AOL" and row.original_label not in forget_labels
        ],
        "unknown_100": [row for row in records if row.source == "100_TOPSEARCH"],
        "full_retain": list(records),
    }
    return {
        name: make_loader(
            rows,
            args.batch_size,
            False,
            args.num_workers,
            device,
            feature_cache_dir,
        )
        for name, rows in slices.items()
        if rows
    }


def evaluate_slices(
    model: FlowModel,
    loaders: dict[str, DataLoader],
    device: torch.device,
    threshold: float,
) -> dict[str, dict[str, object]]:
    return {
        name: evaluate(model, loader, device, threshold)
        for name, loader in loaders.items()
    }


def benchmark_error_row(
    mode: str, repeat: int, metrics: dict[str, object], baseline_total_error: int
) -> dict[str, object]:
    confusion = metrics["confusion_matrix"]
    false_positive = int(confusion[KNOWN][UNKNOWN])
    false_negative = int(confusion[UNKNOWN][KNOWN])
    total_error = false_positive + false_negative
    reduction = (
        100.0 * (baseline_total_error - total_error) / baseline_total_error
        if baseline_total_error > 0
        else float("nan")
    )
    return {
        "mode": mode,
        "repeat": repeat,
        "fn": false_negative,
        "fp": false_positive,
        "total_error": total_error,
        "total_error_reduction_percent": reduction,
    }


def aggregate_benchmark_rows(
    rows: Sequence[dict[str, object]],
) -> list[dict[str, object]]:
    grouped: dict[str, list[dict[str, object]]] = defaultdict(list)
    for row in rows:
        grouped[str(row["mode"])].append(row)
    result: list[dict[str, object]] = []
    for mode, mode_rows in grouped.items():
        aggregate: dict[str, object] = {"mode": mode, "repeats": len(mode_rows)}
        for key in ("fn", "fp", "total_error", "total_error_reduction_percent"):
            values = np.asarray([float(row[key]) for row in mode_rows], dtype=np.float64)
            aggregate[f"{key}_mean"] = float(values.mean())
            aggregate[f"{key}_std"] = float(values.std(ddof=0))
        result.append(aggregate)
    return result


## 8. Pipeline phase implementations


In [ ]:
def inspect_dataset_schema(root: Path, rows: int) -> dict[str, object]:
    mapping = read_label_mapping(root / "label_mapping.csv")
    summary: dict[str, object] = {
        "dataset_dir": str(root),
        "mapping_count": len(mapping),
        "label_zero": mapping.get(0),
        "files": {},
    }
    for file_name in ("train_data.csv", "test_data.csv"):
        path = root / file_name
        inspected: list[dict[str, object]] = []
        with path.open("rb") as handle:
            for line_number in range(1, rows + 1):
                byte_offset = handle.tell()
                raw = handle.readline()
                if not raw:
                    break
                comma = raw.find(b",")
                if comma < 0:
                    raise ValueError(f"Missing comma at {path}:{line_number}")
                label_id = int(float(raw[:comma]))
                if label_id not in mapping:
                    raise ValueError(f"Unknown label_id={label_id} at {path}:{line_number}")
                values = np.fromstring(raw.decode("ascii").strip(), sep=",", dtype=np.float32)
                if values.size != ROW_WIDTH:
                    raise ValueError(
                        f"{path}:{line_number} has {values.size} values; expected {ROW_WIDTH}"
                    )
                packets = values.reshape(MAX_PACKETS, RAW_FIELDS_PER_PACKET)
                real = packets[:, 3] != 0
                if not np.allclose(packets[real, 0], float(label_id)):
                    raise ValueError(f"Inconsistent repeated labels at {path}:{line_number}")
                if not np.allclose(packets[~real], 0.0):
                    raise ValueError(f"Padding must be [0,0,0,0] at {path}:{line_number}")
                inspected.append(
                    {
                        "line_number": line_number,
                        "byte_offset": byte_offset,
                        "label_id": label_id,
                        "label_name": mapping[label_id],
                        "values": int(values.size),
                        "real_packets": int(real.sum()),
                        "padding_packets": int((~real).sum()),
                        "row_sha256": hashlib.sha256(raw.rstrip(b"\r\n")).hexdigest(),
                    }
                )
        if not inspected:
            raise ValueError(f"CSV has no rows: {path}")
        summary["files"][file_name] = {
            "bytes": path.stat().st_size,
            "inspected": inspected,
        }
    return summary


def command_validate_schema(args: argparse.Namespace) -> None:
    aol_dir = ensure_under(Path(args.aol_dir), VM_INPUT_ROOT, "input")
    unknown_dir = ensure_under(Path(args.unknown_dir), VM_INPUT_ROOT, "input")
    preflight_dataset_dirs(aol_dir, unknown_dir)
    output_dir = ensure_under(Path(args.output_dir), VM_WORKING_ROOT, "output")
    output_dir.mkdir(parents=True, exist_ok=True)
    summary = {
        "platform": "virtual_machine",
        "input_root": str(VM_INPUT_ROOT),
        "output_root": str(output_dir),
        "known_aol": inspect_dataset_schema(aol_dir, args.rows),
        "unknown_100_topsearch": inspect_dataset_schema(unknown_dir, args.rows),
        "pretrained_encoder": {
            "path": DEFAULT_PRETRAIN,
            "bytes": Path(DEFAULT_PRETRAIN).stat().st_size,
            "sha256": sha256_file(Path(DEFAULT_PRETRAIN)),
        },
    }
    output_path = output_dir / "schema_validation.json"
    write_json(output_path, summary)
    print(json.dumps(summary, ensure_ascii=False, indent=2))
    print(f"Saved schema validation: {output_path}")


def common_runtime(args: argparse.Namespace) -> tuple[torch.device, Path, Path, Path, Path]:
    device = resolve_device(args.device, args.allow_cpu)
    aol_dir = ensure_under(Path(args.aol_dir), VM_INPUT_ROOT, "input")
    unknown_dir = ensure_under(Path(args.unknown_dir), VM_INPUT_ROOT, "input")
    preflight_dataset_dirs(aol_dir, unknown_dir)
    output_dir = ensure_under(Path(args.output_dir), VM_WORKING_ROOT, "output")
    output_dir.mkdir(parents=True, exist_ok=True)
    feature_cache_dir = ensure_under(
        Path(args.feature_cache_dir), VM_WORKING_ROOT, "output"
    )
    feature_cache_dir.mkdir(parents=True, exist_ok=True)
    index_path = ensure_under(Path(args.index_path), VM_WORKING_ROOT, "input")
    index_path.parent.mkdir(parents=True, exist_ok=True)
    return device, aol_dir, unknown_dir, output_dir, feature_cache_dir


def command_train_base(args: argparse.Namespace) -> None:
    started_at = time.perf_counter()
    set_seed(args.seed)
    device, aol_dir, unknown_dir, output_dir, feature_cache_dir = common_runtime(args)
    source_train, source_test, signatures = load_or_build_index(
        aol_dir, unknown_dir, Path(args.index_path).resolve()
    )
    train, validation, test = split_records(
        source_train, source_test, args.seed, args.valid_ratio
    )
    write_json(
        output_dir / "split_manifest.json",
        {
            "seed": args.seed,
            "valid_ratio": args.valid_ratio,
            "dataset_signatures": signatures,
            "train": [asdict(row) for row in train],
            "validation": [asdict(row) for row in validation],
            "test": [asdict(row) for row in test],
        },
    )
    train_loader = make_loader(
        train, args.batch_size, True, args.num_workers, device, feature_cache_dir
    )
    validation_loader = make_loader(
        validation, args.batch_size, False, args.num_workers, device, feature_cache_dir
    )
    test_loader = make_loader(
        test, args.batch_size, False, args.num_workers, device, feature_cache_dir
    )
    model = FlowModel(dropout=args.dropout).to(device)
    pretrain_path = Path(args.pretrained_encoder).resolve()
    if not pretrain_path.is_file():
        raise FileNotFoundError(pretrain_path)
    pretrain_info = load_pretrained_encoder(model, pretrain_path)
    best_state, history = train_base_model(
        model,
        train_loader,
        validation_loader,
        train,
        device,
        args.epochs,
        args.learning_rate,
        args.weight_decay,
        freeze_encoder=not args.fine_tune_encoder,
    )
    model.load_state_dict(best_state)
    threshold, threshold_rows = select_threshold(
        model,
        validation_loader,
        device,
        tuple(float(value) for value in parse_csv_values(args.threshold_grid)),
    )
    test_metrics = evaluate(model, test_loader, device, threshold)
    checkpoint_path = output_dir / "base_model" / "best_model.pt"
    checkpoint_path.parent.mkdir(parents=True, exist_ok=True)
    checkpoint = {
        "model_state_dict": state_to_cpu(model.state_dict()),
        "model_config": {
            "max_packets": MAX_PACKETS,
            "packet_features": PACKET_FEATURES,
            "embedding_dim": EMBEDDING_DIM,
            "hidden_dims": [512, 256, 128, 64, 32],
            "dropout": args.dropout,
            "feature_transform": FEATURE_TRANSFORM,
        },
        "best_unknown_threshold": threshold,
        "unknown_threshold": threshold,
        "split_ids": split_ids_payload(train, validation, test),
        "dataset_signatures": signatures,
        "seed": args.seed,
        "valid_ratio": args.valid_ratio,
        "pretrain": pretrain_info,
        "test_metrics": test_metrics,
    }
    torch.save(checkpoint, checkpoint_path)
    write_json(
        output_dir / "base_model" / "result.json",
        {
            "checkpoint": str(checkpoint_path),
            "checkpoint_sha256": sha256_file(checkpoint_path),
            "pretrain": pretrain_info,
            "dataset_signatures": signatures,
            "runtime_seconds": time.perf_counter() - started_at,
            "counts": {
                "train": len(train),
                "validation": len(validation),
                "test": len(test),
                "train_binary": dict(Counter(row.binary_label for row in train)),
            },
            "history": history,
            "threshold_sweep": threshold_rows,
            "best_unknown_threshold": threshold,
            "test": test_metrics,
        },
    )
    print(f"Saved base checkpoint: {checkpoint_path}")
    print("Base test:", test_metrics)


def command_make_manifests(args: argparse.Namespace) -> None:
    set_seed(args.seed)
    device = resolve_device(args.device, args.allow_cpu)
    model, checkpoint, train, _, _, signatures = load_base_context(args, device)
    feature_cache_dir = Path(args.feature_cache_dir).resolve()
    aol_train = [row for row in train if row.source == "AOL"]
    loader = make_loader(
        aol_train, args.batch_size, False, args.num_workers, device, feature_cache_dir
    )
    threshold = float(
        checkpoint.get("best_unknown_threshold", checkpoint.get("unknown_threshold", 0.5))
    )
    scores = {row["instance_id"]: row for row in score_records(model, loader, device, threshold)}
    eligible = [
        row for row in aol_train if scores[row.instance_id]["prediction"] == KNOWN
    ]
    ordered = stratified_nested_order(eligible, args.selection_seed)
    counts = parse_int_values(args.counts)
    if any(count <= 0 or count > len(ordered) for count in counts):
        raise ValueError(f"Forget counts phải nằm trong [1, {len(ordered)}]: {counts}")
    output_dir = Path(args.manifest_output_dir).resolve()
    output_dir.mkdir(parents=True, exist_ok=True)
    for count in counts:
        selected = ordered[:count]
        manifest = {
            "version": 1,
            "selection": "nested_stratified_correctly_known",
            "selection_seed": args.selection_seed,
            "forget_count": count,
            "target_binary_label": UNKNOWN,
            "base_checkpoint": checkpoint["checkpoint_path"],
            "base_checkpoint_sha256": checkpoint["checkpoint_sha256"],
            "dataset_signatures": signatures,
            "instances": [
                {
                    "instance_id": row.instance_id,
                    "source": row.source,
                    "split_file": row.split_file,
                    "line_number": row.line_number,
                    "byte_offset": row.byte_offset,
                    "row_sha256": row.row_sha256,
                    "original_label": row.original_label,
                    "original_label_id": row.original_label_id,
                    "probability_unknown_before": scores[row.instance_id][
                        "probability_unknown"
                    ],
                }
                for row in selected
            ],
        }
        path = output_dir / f"forget_{count}.json"
        write_json(path, manifest)
        print(f"Saved {path}")


# Legacy method-by-scope command retained for old artifacts; it is not registered.
def _legacy_command_unlearn(args: argparse.Namespace) -> None:
    command_started_at = time.perf_counter()
    device = resolve_device(args.device, args.allow_cpu)
    base_model, checkpoint, train, validation, test, signatures = load_base_context(
        args, device
    )
    threshold = float(
        checkpoint.get("best_unknown_threshold", checkpoint.get("unknown_threshold", 0.5))
    )
    manifest_path = Path(args.forget_manifest).resolve()
    manifest, forget_records = validate_forget_manifest(manifest_path, train)
    manifest_checkpoint_hash = manifest.get("base_checkpoint_sha256")
    if manifest_checkpoint_hash and manifest_checkpoint_hash != checkpoint["checkpoint_sha256"]:
        raise ValueError("Forget manifest được tạo từ base checkpoint khác")
    manifest_signatures = manifest.get("dataset_signatures")
    if manifest_signatures and manifest_signatures != signatures:
        raise ValueError("Forget manifest được tạo từ dataset signatures khác")

    forget_ids = {row.instance_id for row in forget_records}
    retain_train = [row for row in train if row.instance_id not in forget_ids]
    if len(forget_records) + len(retain_train) != len(train):
        raise AssertionError("Df union Dr phải bằng base train")
    forget_labels = {row.original_label for row in forget_records}
    feature_cache_dir = Path(args.feature_cache_dir).resolve()
    forget_loader = make_loader(
        forget_records,
        args.batch_size,
        True,
        args.num_workers,
        device,
        feature_cache_dir,
    )
    forget_eval_loader = make_loader(
        forget_records,
        args.batch_size,
        False,
        args.num_workers,
        device,
        feature_cache_dir,
    )
    retain_loader = make_loader(
        retain_train,
        args.batch_size,
        True,
        args.num_workers,
        device,
        feature_cache_dir,
    )
    validation_loaders = build_slice_loaders(
        validation, forget_labels, args, device, feature_cache_dir
    )
    test_loaders = build_slice_loaders(test, forget_labels, args, device, feature_cache_dir)

    before = {
        "forget": forget_metrics(base_model, forget_eval_loader, device, threshold),
        "validation": evaluate_slices(base_model, validation_loaders, device, threshold),
        "test": evaluate_slices(base_model, test_loaders, device, threshold),
    }
    methods = parse_csv_values(args.methods)
    scopes = parse_csv_values(args.scopes)
    unknown_methods = set(methods) - set(LEGACY_METHODS)
    unknown_scopes = set(scopes) - set(TRAINABLE_SCOPES)
    if unknown_methods:
        raise ValueError(f"Method không hợp lệ: {sorted(unknown_methods)}")
    if unknown_scopes:
        raise ValueError(f"Scope không hợp lệ: {sorted(unknown_scopes)}")

    output_root = Path(args.output_dir).resolve() / "instance_unlearning" / manifest_path.stem
    output_root.mkdir(parents=True, exist_ok=True)
    pgd_config = PgdConfig(
        steps=args.pgd_steps,
        epsilon_time=args.pgd_epsilon_time,
        epsilon_size=args.pgd_epsilon_size,
        alpha_time=args.pgd_alpha_time,
        alpha_size=args.pgd_alpha_size,
        max_packet_size=args.max_packet_size,
        round_packet_size=not args.no_round_packet_size,
    )
    adversarial_payload: dict[str, Any] | None = None
    adversarial_path: Path | None = None
    if any(method in ("l2ul_adv", "l2ul_adv_mas") for method in methods):
        # A non-shuffled loader keeps IDs aligned with cached tensors.
        attack_loader = make_loader(
            forget_records,
            args.batch_size,
            False,
            args.num_workers,
            device,
            feature_cache_dir,
        )
        adversarial_payload, adversarial_path = load_or_generate_adversarial(
            base_model,
            forget_records,
            attack_loader,
            checkpoint["checkpoint_sha256"],
            output_root / "adversarial_cache",
            device,
            pgd_config,
        )

    summary: dict[str, Any] = {
        "base_checkpoint": checkpoint["checkpoint_path"],
        "base_checkpoint_sha256": checkpoint["checkpoint_sha256"],
        "forget_manifest": str(manifest_path),
        "forget_manifest_sha256": sha256_file(manifest_path),
        "dataset_signatures": signatures,
        "methods": list(methods),
        "scopes": list(scopes),
        "repeats": args.repeats,
        "forget_count": len(forget_records),
        "forget_instance_ids": [row.instance_id for row in forget_records],
        "forget_original_labels": sorted(forget_labels),
        "threshold": threshold,
        "before": before,
        "pgd_config": asdict(pgd_config),
        "adversarial_cache": str(adversarial_path) if adversarial_path else None,
        "experiments": [],
    }

    for method in methods:
        for scope in scopes:
            for repeat in range(args.repeats):
                optimization_seed = args.optimization_seed + repeat
                set_seed(optimization_seed)
                print(
                    f"Running method={method} scope={scope} repeat={repeat} "
                    f"seed={optimization_seed}"
                )
                branch_started_at = time.perf_counter()
                branch, history, anchors = _legacy_run_unlearning_method(
                    base_model,
                    method,
                    scope,
                    forget_loader,
                    retain_loader,
                    adversarial_payload,
                    validation_loaders,
                    device,
                    threshold,
                    args.unlearning_epochs,
                    args.unlearning_learning_rate,
                    args.unlearning_weight_decay,
                    args.lambda_retain,
                    args.lambda_adv,
                    args.lambda_mas,
                    args.batch_size,
                    args.max_steps_per_epoch,
                )
                after_forget = forget_metrics(
                    branch, forget_eval_loader, device, threshold
                )
                after_validation = evaluate_slices(
                    branch, validation_loaders, device, threshold
                )
                after_test = evaluate_slices(branch, test_loaders, device, threshold)
                directory = output_root / method / scope / f"repeat_{repeat}"
                directory.mkdir(parents=True, exist_ok=True)
                model_path = directory / "unlearned_model.pt"
                torch.save(
                    {
                        "model_state_dict": state_to_cpu(branch.state_dict()),
                        "model_config": checkpoint.get("model_config", {}),
                        "base_checkpoint_sha256": checkpoint["checkpoint_sha256"],
                        "forget_manifest": str(manifest_path),
                        "method": method,
                        "scope": scope,
                        "repeat": repeat,
                        "optimization_seed": optimization_seed,
                        "unknown_threshold": threshold,
                    },
                    model_path,
                )
                if anchors is not None:
                    torch.save(
                        {name: value.detach().cpu() for name, value in anchors.items()},
                        directory / "mas_anchors.pt",
                    )
                result = {
                    "method": method,
                    "scope": scope,
                    "repeat": repeat,
                    "optimization_seed": optimization_seed,
                    "runtime_seconds": time.perf_counter() - branch_started_at,
                    "base_checkpoint_sha256": checkpoint["checkpoint_sha256"],
                    "forget_manifest": str(manifest_path),
                    "forget_manifest_sha256": sha256_file(manifest_path),
                    "dataset_signatures": signatures,
                    "forget_instance_ids": [row.instance_id for row in forget_records],
                    "forget_original_labels": sorted(forget_labels),
                    "model": str(model_path),
                    "before_forget": before["forget"],
                    "before_validation": before["validation"],
                    "before_test": before["test"],
                    "after_forget": after_forget,
                    "after_validation": after_validation,
                    "after_test": after_test,
                    "history": history,
                }
                write_json(directory / "result.json", result)
                summary["experiments"].append(result)
    summary["runtime_seconds"] = time.perf_counter() - command_started_at
    write_json(output_root / "summary.json", summary)
    print(f"Saved unlearning summary: {output_root / 'summary.json'}")


def command_unlearn_modes(args: argparse.Namespace) -> None:
    command_started_at = time.perf_counter()
    device = resolve_device(args.device, args.allow_cpu)
    base_model, checkpoint, train, validation, test, signatures = load_base_context(
        args, device
    )
    threshold = float(
        checkpoint.get("best_unknown_threshold", checkpoint.get("unknown_threshold", 0.5))
    )
    manifest_path = Path(args.forget_manifest).resolve()
    manifest, forget_records = validate_forget_manifest(manifest_path, train)
    manifest_checkpoint_hash = manifest.get("base_checkpoint_sha256")
    if manifest_checkpoint_hash and manifest_checkpoint_hash != checkpoint["checkpoint_sha256"]:
        raise ValueError("Forget manifest được tạo từ base checkpoint khác")
    manifest_signatures = manifest.get("dataset_signatures")
    if manifest_signatures and manifest_signatures != signatures:
        raise ValueError("Forget manifest được tạo từ dataset signatures khác")

    forget_ids = {row.instance_id for row in forget_records}
    retain_train = [row for row in train if row.instance_id not in forget_ids]
    unknown_reference_records = [
        row for row in retain_train if row.binary_label == UNKNOWN
    ]
    if len(forget_records) + len(retain_train) != len(train):
        raise AssertionError("Df union Dr phải bằng base train")
    if not unknown_reference_records:
        raise ValueError("Base train không có unknown reference cho MKMMD")

    modes = parse_csv_values(args.modes)
    unknown_modes = set(modes) - set(MODE_SPECS)
    if unknown_modes:
        raise ValueError(f"Mode không hợp lệ: {sorted(unknown_modes)}")
    if args.repeats < 1:
        raise ValueError("repeats must be >= 1")
    mode_hyperparameters = parse_mode_hyperparameters(args.mode_config_json, modes)

    forget_labels = {row.original_label for row in forget_records}
    feature_cache_dir = Path(args.feature_cache_dir).resolve()
    forget_loader = make_loader(
        forget_records, args.batch_size, True, args.num_workers, device, feature_cache_dir
    )
    forget_eval_loader = make_loader(
        forget_records, args.batch_size, False, args.num_workers, device, feature_cache_dir
    )
    retain_loader = make_loader(
        retain_train, args.batch_size, True, args.num_workers, device, feature_cache_dir
    )
    unknown_reference_loader = make_loader(
        unknown_reference_records,
        args.batch_size,
        True,
        args.num_workers,
        device,
        feature_cache_dir,
    )
    validation_loaders = build_slice_loaders(
        validation, forget_labels, args, device, feature_cache_dir
    )
    test_loaders = build_slice_loaders(
        test, forget_labels, args, device, feature_cache_dir
    )

    before = {
        "forget": forget_metrics(base_model, forget_eval_loader, device, threshold),
        "validation": evaluate_slices(base_model, validation_loaders, device, threshold),
        "test": evaluate_slices(base_model, test_loaders, device, threshold),
    }
    baseline_confusion = before["test"]["full_retain"]["confusion_matrix"]
    baseline_total_error = int(
        baseline_confusion[KNOWN][UNKNOWN] + baseline_confusion[UNKNOWN][KNOWN]
    )
    output_root = (
        Path(args.output_dir).resolve() / "instance_unlearning" / manifest_path.stem
    )
    output_root.mkdir(parents=True, exist_ok=True)
    summary: dict[str, Any] = {
        "base_checkpoint": checkpoint["checkpoint_path"],
        "base_checkpoint_sha256": checkpoint["checkpoint_sha256"],
        "forget_manifest": str(manifest_path),
        "forget_manifest_sha256": sha256_file(manifest_path),
        "dataset_signatures": signatures,
        "modes": list(modes),
        "mode_specs": {mode: MODE_SPECS[mode] for mode in modes},
        "mode_hyperparameters": {
            mode: asdict(mode_hyperparameters[mode]) for mode in modes
        },
        "repeats": args.repeats,
        "save_models": args.save_models,
        "forget_count": len(forget_records),
        "forget_instance_ids": [row.instance_id for row in forget_records],
        "forget_original_labels": sorted(forget_labels),
        "threshold": threshold,
        "before": before,
        "experiments": [],
    }
    benchmark_rows: list[dict[str, object]] = []

    for mode in modes:
        mode_repeats = 1 if mode == "baseline" else args.repeats
        for repeat in range(mode_repeats):
            optimization_seed = args.optimization_seed + repeat
            set_seed(optimization_seed)
            spec = MODE_SPECS[mode]
            mode_config = mode_hyperparameters[mode]
            print(
                f"Running mode={mode} repeat={repeat} seed={optimization_seed} "
                f"scope={spec['trainable_scope']} epochs={mode_config.epochs} "
                f"lr={mode_config.learning_rate:g} "
                f"lambda_forget={mode_config.lambda_forget:g} "
                f"lambda_retain={mode_config.lambda_retain:g}"
            )
            branch_started_at = time.perf_counter()
            branch, history = run_unlearning_mode(
                base_model,
                mode,
                forget_loader,
                retain_loader,
                unknown_reference_loader,
                validation_loaders,
                device,
                threshold,
                mode_config,
            )
            after_forget = forget_metrics(
                branch, forget_eval_loader, device, threshold
            )
            after_validation = evaluate_slices(
                branch, validation_loaders, device, threshold
            )
            after_test = evaluate_slices(branch, test_loaders, device, threshold)
            directory = output_root / mode / f"repeat_{repeat}"
            directory.mkdir(parents=True, exist_ok=True)
            model_path: Path | None = None
            if args.save_models and mode != "baseline":
                model_path = directory / "unlearned_model.pt"
                torch.save(
                    {
                        "model_state_dict": state_to_cpu(branch.state_dict()),
                        "model_config": checkpoint.get("model_config", {}),
                        "base_checkpoint_sha256": checkpoint["checkpoint_sha256"],
                        "forget_manifest": str(manifest_path),
                        "mode": mode,
                        "mode_spec": spec,
                        "mode_hyperparameters": asdict(mode_config),
                        "repeat": repeat,
                        "optimization_seed": optimization_seed,
                        "unknown_threshold": threshold,
                    },
                    model_path,
                )
            result = {
                "mode": mode,
                "mode_spec": spec,
                "mode_hyperparameters": asdict(mode_config),
                "repeat": repeat,
                "optimization_seed": optimization_seed,
                "runtime_seconds": time.perf_counter() - branch_started_at,
                "base_checkpoint_sha256": checkpoint["checkpoint_sha256"],
                "forget_manifest": str(manifest_path),
                "forget_manifest_sha256": sha256_file(manifest_path),
                "dataset_signatures": signatures,
                "forget_instance_ids": [row.instance_id for row in forget_records],
                "forget_original_labels": sorted(forget_labels),
                "model": str(model_path) if model_path else None,
                "before_forget": before["forget"],
                "before_validation": before["validation"],
                "before_test": before["test"],
                "after_forget": after_forget,
                "after_validation": after_validation,
                "after_test": after_test,
                "history": history,
            }
            write_json(directory / "result.json", result)
            summary["experiments"].append(result)
            benchmark_rows.append(
                benchmark_error_row(
                    mode, repeat, after_test["full_retain"], baseline_total_error
                )
            )
            del branch
            if device.type == "cuda":
                torch.cuda.empty_cache()

    benchmark_summary = aggregate_benchmark_rows(benchmark_rows)
    summary["benchmark_rows"] = benchmark_rows
    summary["benchmark_summary"] = benchmark_summary
    summary["runtime_seconds"] = time.perf_counter() - command_started_at
    write_csv_rows(output_root / "benchmark_runs.csv", benchmark_rows)
    write_csv_rows(output_root / "benchmark_summary.csv", benchmark_summary)
    write_json(output_root / "summary.json", summary)
    print(f"Saved mode benchmark: {output_root / 'benchmark_summary.csv'}")


## 9. Argument definitions shared by notebook phases


In [ ]:
def add_runtime_arguments(parser: argparse.ArgumentParser) -> None:
    parser.add_argument("--aol-dir", default=os.environ.get("AOL_DATASET_DIR", DEFAULT_AOL_DIR))
    parser.add_argument(
        "--unknown-dir",
        default=os.environ.get("UNKNOWN_100_DATASET_DIR", DEFAULT_UNKNOWN_DIR),
    )
    parser.add_argument("--output-dir", default=str(OUTPUT_ROOT / "base_run"))
    parser.add_argument("--feature-cache-dir", default=str(OUTPUT_ROOT / "feature_cache"))
    parser.add_argument("--index-path", default=str(OUTPUT_ROOT / "csv_index.json"))
    parser.add_argument("--batch-size", type=int, default=16)
    parser.add_argument("--num-workers", type=int, default=2)
    parser.add_argument("--seed", type=int, default=42)
    parser.add_argument("--valid-ratio", type=float, default=0.15)
    parser.add_argument("--device", default="cuda:0")
    parser.add_argument("--allow-cpu", action="store_true")


def build_parser() -> argparse.ArgumentParser:
    parser = argparse.ArgumentParser(description=__doc__)
    subparsers = parser.add_subparsers(dest="command", required=True)

    validate = subparsers.add_parser(
        "validate-schema", help="Resolve and validate both local CSV inputs"
    )
    validate.add_argument("--aol-dir", default=DEFAULT_AOL_DIR)
    validate.add_argument("--unknown-dir", default=DEFAULT_UNKNOWN_DIR)
    validate.add_argument("--output-dir", default=str(OUTPUT_ROOT))
    validate.add_argument("--rows", type=int, default=2)
    validate.set_defaults(handler=command_validate_schema)

    train = subparsers.add_parser("train-base", help="Train encoder+MLP binary base model")
    add_runtime_arguments(train)
    train.add_argument("--pretrained-encoder", default=DEFAULT_PRETRAIN)
    train.add_argument("--epochs", type=int, default=16)
    train.add_argument("--learning-rate", type=float, default=2e-4)
    train.add_argument("--weight-decay", type=float, default=1e-4)
    train.add_argument("--dropout", type=float, default=0.2)
    train.add_argument("--fine-tune-encoder", action="store_true")
    train.add_argument(
        "--threshold-grid",
        default="0.30,0.35,0.40,0.45,0.50,0.55,0.60,0.65,0.70",
    )
    train.set_defaults(handler=command_train_base)

    manifests = subparsers.add_parser(
        "make-manifests", help="Create nested forget manifests from AOL base-train"
    )
    add_runtime_arguments(manifests)
    manifests.add_argument("--base-checkpoint", required=True)
    manifests.add_argument("--counts", default="1,4,10,50,100")
    manifests.add_argument("--selection-seed", type=int, default=42)
    manifests.add_argument(
        "--manifest-output-dir", default=str(OUTPUT_ROOT / "forget_manifests")
    )
    manifests.set_defaults(handler=command_make_manifests)

    unlearn = subparsers.add_parser("unlearn", help="Run instance-wise unlearning")
    add_runtime_arguments(unlearn)
    unlearn.add_argument("--base-checkpoint", required=True)
    unlearn.add_argument("--forget-manifest", required=True)
    unlearn.add_argument("--modes", default=",".join(DEFAULT_BENCHMARK_MODES))
    unlearn.add_argument("--repeats", type=int, default=3)
    unlearn.add_argument("--optimization-seed", type=int, default=42)
    unlearn.add_argument(
        "--mode-config-json",
        default=json.dumps(DEFAULT_MODE_HYPERPARAMETERS),
        help="JSON object containing an independent optimization preset per mode",
    )
    unlearn.add_argument("--save-models", action="store_true")
    unlearn.set_defaults(handler=command_unlearn_modes)
    return parser


def main() -> None:
    args = build_parser().parse_args()
    args.handler(args)


## 10. Notebook phase configuration


In [ ]:
# OVERVIEW: Configure the VM phases executed by the final cell.
# Keep False while reviewing paths and configuration.
RUN_PIPELINE = True

VM_AOL_DATASET_DIR = Path(DEFAULT_AOL_DIR)
VM_UNKNOWN_100_DATASET_DIR = Path(DEFAULT_UNKNOWN_DIR)
VM_PRETRAIN_AOL_PATH = Path(DEFAULT_PRETRAIN)
VM_OUTPUT_ROOT = OUTPUT_ROOT
VM_FEATURE_CACHE = VM_OUTPUT_ROOT / "feature_cache"
VM_INDEX_PATH = VM_OUTPUT_ROOT / "csv_index.json"
VM_BASE_CHECKPOINT = VM_OUTPUT_ROOT / "base_run" / "base_model" / "best_model.pt"
VM_MANIFEST_DIR = VM_OUTPUT_ROOT / "forget_manifests"
FORGET_COUNT = 1

NOTEBOOK_ARGUMENTS_BY_COMMAND = {
    "validate-schema": [
        "--aol-dir", str(VM_AOL_DATASET_DIR),
        "--unknown-dir", str(VM_UNKNOWN_100_DATASET_DIR),
        "--output-dir", str(VM_OUTPUT_ROOT),
        "--rows", "2",
    ],
    "train-base": [
        "--aol-dir", str(VM_AOL_DATASET_DIR),
        "--unknown-dir", str(VM_UNKNOWN_100_DATASET_DIR),
        "--pretrained-encoder", str(VM_PRETRAIN_AOL_PATH),
        "--output-dir", str(VM_OUTPUT_ROOT / "base_run"),
        "--feature-cache-dir", str(VM_FEATURE_CACHE),
        "--index-path", str(VM_INDEX_PATH),
        "--epochs", "16",
        "--device", "cuda:0",
    ],
    "make-manifests": [
        "--aol-dir", str(VM_AOL_DATASET_DIR),
        "--unknown-dir", str(VM_UNKNOWN_100_DATASET_DIR),
        "--base-checkpoint", str(VM_BASE_CHECKPOINT),
        "--counts", "1,4,10,50,100",
        "--selection-seed", "42",
        "--manifest-output-dir", str(VM_MANIFEST_DIR),
        "--output-dir", str(VM_OUTPUT_ROOT / "base_run"),
        "--feature-cache-dir", str(VM_FEATURE_CACHE),
        "--index-path", str(VM_INDEX_PATH),
        "--device", "cuda:0",
    ],
    "unlearn": [
        "--aol-dir", str(VM_AOL_DATASET_DIR),
        "--unknown-dir", str(VM_UNKNOWN_100_DATASET_DIR),
        "--base-checkpoint", str(VM_BASE_CHECKPOINT),
        "--forget-manifest", str(VM_MANIFEST_DIR / f"forget_{FORGET_COUNT}.json"),
        "--modes", "baseline,only_head,full_mlp,last_encoder_block,full_model,amnesiac,blindspot,bnd,mkmmd",
        "--repeats", "3",
        "--mode-config-json", json.dumps(DEFAULT_MODE_HYPERPARAMETERS),
        "--output-dir", str(VM_OUTPUT_ROOT / "base_run"),
        "--feature-cache-dir", str(VM_FEATURE_CACHE),
        "--index-path", str(VM_INDEX_PATH),
        "--device", "cuda:0",
    ],
}

disk = shutil.disk_usage(VM_WORKING_ROOT)
print("VM known input:", VM_AOL_DATASET_DIR)
print("VM unknown input:", VM_UNKNOWN_100_DATASET_DIR)
print("Encoder checkpoint:", VM_PRETRAIN_AOL_PATH)
print("Output root:", VM_OUTPUT_ROOT)
print("CUDA available:", torch.cuda.is_available())
print("VM working free GiB:", round(disk.free / (1024 ** 3), 2))
print("RUN_PIPELINE:", RUN_PIPELINE)


## 11. Execute configured phases sequentially


In [ ]:
# OVERVIEW: Execute the selected notebook phases sequentially.
NOTEBOOK_COMMANDS = [
    "validate-schema",
    "train-base",
    "make-manifests",
    "unlearn",
]

unknown_commands = set(NOTEBOOK_COMMANDS) - set(NOTEBOOK_ARGUMENTS_BY_COMMAND)
if unknown_commands:
    raise ValueError(f"Unknown notebook commands: {sorted(unknown_commands)}")

if RUN_PIPELINE:
    for command_index, notebook_command in enumerate(NOTEBOOK_COMMANDS, start=1):
        print(
            f"\n[{command_index}/{len(NOTEBOOK_COMMANDS)}] "
            f"Running notebook command: {notebook_command}"
        )
        notebook_args = build_parser().parse_args(
            [notebook_command, *NOTEBOOK_ARGUMENTS_BY_COMMAND[notebook_command]]
        )
        notebook_args.handler(notebook_args)

    print("\nAll notebook commands completed successfully.")
else:
    print("Pipeline is disabled. Review the config, then set RUN_PIPELINE=True.")
